# PyTorch 模块全解 · 练习题册

配套手册：**《PyTorch 模块全解：API 与神经网络知识点的对照手册》**

把手册 21 章的内容压成 **57 道可自动判分的练习**：每道题先自己写代码或做选择，再运行「自检」格，看到 `[PASS]` 才过关；文末附全部参考答案。

覆盖范围：张量 → autograd → Module → CNN / RNN → 归一化与激活 → 注意力 → 损失 → 优化器与调度 → 工程工具 → 综合实战。

## 怎么用这本册子

1. **内核**：选 conda 环境 `swift_env`（torch 2.6）。没有 GPU 也能全部跑通，涉及 CUDA 的题会自动退回 CPU。
2. **顺序**：从上往下做，后面的题默认你会前面用到的写法；每题都标了难度与对应手册章节。
3. **判分**：改完 starter 代码后，运行它下面的「自检」格。`[FAIL]` 会指出具体差在哪一项，常见原因有三种：
   - 变量名没对上（题目要求填 `shapes_1_2`，你写成了 `shape`）；
   - 形状 / dtype / 数值确实不对；
   - 上一格没运行（报 `NameError`，变量还不存在）。
4. **卡住**：先看题目的「提示」；超过 10 分钟再看文末「参考答案」。
5. **计分**：每部分末尾跑一次 `scoreboard()`，查看累计通过数。
6. **不要用「全部运行」来自检**：未完成的题都会报 `[FAIL]`，这是正常的；请按题逐个运行 starter 与自检格。

## 难度标记

| 标记 | 含义 |
| --- | --- |
| ★ | 会查 API 就能做 |
| ★★ | 需要理解形状与语义 |
| ★★★ | 需要自己推导或组合多个 API |

## 分部地图

| 部分 | 主题 | 对应手册 |
| --- | --- | --- |
| 1 | 张量与运算 | 第 1–2 章 |
| 2 | 自动求导 | 第 3 章 |
| 3 | Module 与线性层 | 第 4–5 章 |
| 4 | 卷积与池化 | 第 6 章 |
| 5 | 循环网络 | 第 7 章 |
| 6 | 归一化、激活、Dropout | 第 8–9 章 |
| 7 | Embedding 与注意力 | 第 10 章 |
| 8 | 损失函数 | 第 11 章 |
| 9 | 优化器与学习率 | 第 12–13 章 |
| 10 | 工程工具 | 第 14–19 章 |
| 11 | 综合实战 | 第 20 章 + 附录 |

## 判分约定

- `check(name, fn)`：`fn` 返回 True 通过；返回 False 或抛异常都算失败，并打印异常类型（`NameError` 通常表示上一格还没运行）。
- `quiz(name, given, answer)`：选择题，把选项字母填进变量后再运行自检。
- `close(a, b)`：`torch.allclose` 的包装，默认容差 1e-5。
- `scoreboard()`：打印累计通过 / 未通过数量。

> 自检失败是学习的一部分：它告诉你「差在哪一项」，而不是「你错了」。改完再跑一次即可。

In [ ]:
# 环境准备：导入依赖 + 自动判分工具（请直接运行，不要修改）
import copy
import math
import os
import tempfile

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("device :", device)

# ---------- 判分工具 ----------
_stats = {"pass": 0, "fail": 0}


def check(name, fn):
    """fn() 返回 True 视为通过；返回 False 或抛异常都记为失败。"""
    try:
        ok, note = bool(fn()), ""
    except Exception as exc:
        ok, note = False, f"  <{type(exc).__name__}: {exc}>"
    _stats["pass" if ok else "fail"] += 1
    print(("[PASS] " if ok else "[FAIL] ") + name + note)


def quiz(name, given, answer, explain=""):
    ok = given == answer
    _stats["pass" if ok else "fail"] += 1
    print(("[PASS] " if ok else "[FAIL] ") + name
          + (f"  | 解析：{explain}" if (not ok and explain) else ""))


def close(a, b, atol=1e-5):
    a = torch.as_tensor(a).detach().float().cpu()
    b = torch.as_tensor(b).detach().float().cpu()
    return a.shape == b.shape and torch.allclose(a, b, atol=atol, rtol=1e-4)


def scoreboard():
    total = _stats["pass"] + _stats["fail"]
    print(f"已检查 {total} 项：通过 {_stats['pass']}，未通过 {_stats['fail']}")

## 第一部分　张量基础（手册第 1–2 章）

这一部分只练三句话：**建张量先想 dtype，改形状先想连续，cat 不加维、stack 才加维**。
练习编号与手册章号无关，每题都标了「对应手册」。
建议用时 60–90 分钟，全部可在 CPU 上完成。

### 练习 1.1　dtype 陷阱　★

**对应手册**：1.2 创建 / 第 5 章 nn.Linear

`torch.tensor([[1, 2], [3, 4]])` 是从 Python 整数创建的张量。

**要做什么**
1. 把它的 `dtype` 赋给 `dtype_1_1`（直接取属性，不要转换）。
2. 把它转成模型能接受的 `float32`，结果赋给 `t_float`。
3. 用 `try/except` 实测 `nn.Linear(2, 2)` 能否直接接收这个整数张量，把布尔结果赋给 `linear_accepts_int`。

**提示**：整数默认不是 float；`nn.Linear` 的权重是 float32，矩阵乘要求两边 dtype 一致。

In [ ]:
t_int = torch.tensor([[1, 2], [3, 4]])

dtype_1_1 = None          # TODO: t_int.dtype 是什么？
t_float = None            # TODO: 转成 float32
linear_accepts_int = None # TODO: try/except 实测后填 True / False

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("1.1a 整数张量默认 dtype 是 int64", lambda: dtype_1_1 == torch.int64)
check("1.1b 已转为 float32 且数值不变", lambda: t_float.dtype == torch.float32 and close(t_float, t_int))
check("1.1c nn.Linear 不接受 int 输入", lambda: linear_accepts_int is False)

### 练习 1.2　形状预测　★

**对应手册**：1.3 形状操作

已知 `x = torch.randn(2, 3, 4)`。**先手算**下面六个表达式的形状，再填进字典 `shapes_1_2`（值必须写成元组）：

| 键 | 表达式 |
| --- | --- |
| `view` | `x.view(2, -1)` |
| `permute` | `x.permute(2, 0, 1)` |
| `transpose` | `x.transpose(1, 2)` |
| `unsqueeze` | `x.unsqueeze(0)` |
| `flatten` | `x.flatten(1)` |
| `stack` | `torch.stack([x, x], dim=1)` |

**提示**：`permute` / `transpose` 只换轴顺序，元素总数不变；`unsqueeze` 在第 0 维插一个 1；`flatten(1)` 把第 1 维及之后全部压平。

In [ ]:
x_1_2 = torch.randn(2, 3, 4)

shapes_1_2 = {
    "view": None,       # TODO
    "permute": None,    # TODO
    "transpose": None,  # TODO
    "unsqueeze": None,  # TODO
    "flatten": None,    # TODO
    "stack": None,      # TODO
}

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
_expect_1_2 = {
    "view": (2, 12),
    "permute": (4, 2, 3),
    "transpose": (2, 4, 3),
    "unsqueeze": (1, 2, 3, 4),
    "flatten": (2, 12),
    "stack": (2, 2, 3, 4),
}
for _k, _v in _expect_1_2.items():
    check(f"1.2 {_k:<9s} 期望 {_v}", lambda _k=_k, _v=_v: shapes_1_2[_k] == _v)

### 练习 1.3　view 报错与修复　★

**对应手册**：1.3 形状操作 / 1.5 视图与拷贝

```python
x = torch.randn(2, 3, 4)
y = x.permute(1, 2, 0)      # (3, 4, 2)，但内存不连续
z = y.view(3, 8)            # 这一行会报错
```

**要做什么**
1. 复现这个错误，把异常的类型名（字符串，例如 `"RuntimeError"`）赋给 `err_1_3`。
2. 修复它：得到形状 `(3, 8)` 且数值与 `y.reshape(3, 8)` 一致的结果 `z`。

**提示**：`view` 要求内存连续，`permute` 之后不连续；`contiguous()` 或直接用 `reshape` 都可以。

In [ ]:
x_1_3 = torch.randn(2, 3, 4)
y_1_3 = x_1_3.permute(1, 2, 0)

err_1_3 = None   # TODO: 复现 y.view(3, 8)，填 type(e).__name__
z_1_3 = None     # TODO: 修复后得到 (3, 8)

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("1.3a 报错类型为 RuntimeError", lambda: err_1_3 == "RuntimeError")
check("1.3b z 形状为 (3,8)", lambda: tuple(z_1_3.shape) == (3, 8))
check("1.3c z 数值等于 y.reshape(3,8)", lambda: close(z_1_3, y_1_3.reshape(3, 8)))

### 练习 1.4　cat 与 stack　★

**对应手册**：1.4 拼接与切分

`a`、`b` 都是形状 `(2, 3)` 的张量。

**要做什么**
1. 沿第 0 维**拼接**得到 `c_1_4`，形状应为 `(4, 3)`。
2. 沿第 0 维**堆叠**得到 `s_1_4`，形状应为 `(2, 2, 3)`。
3. 沿**最后一维**堆叠得到 `s_last_1_4`，形状应为 `(2, 3, 2)`。
4. 把三个形状填进 `shapes_1_4`。

**提示**：`cat` 不新增维度，`stack` 一定新增一个维度——`stack` 的 `dim` 指的是「新维插在哪」。

In [ ]:
a_1_4 = torch.randn(2, 3)
b_1_4 = torch.randn(2, 3)

c_1_4 = None       # TODO: 沿 dim=0 拼接
s_1_4 = None       # TODO: 沿 dim=0 堆叠
s_last_1_4 = None  # TODO: 沿最后一维堆叠

shapes_1_4 = {"c": None, "s": None, "s_last": None}  # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("1.4a c 形状 (4,3) 且内容正确",
      lambda: tuple(c_1_4.shape) == (4, 3) and close(c_1_4[:2], a_1_4) and close(c_1_4[2:], b_1_4))
check("1.4b s 形状 (2,2,3) 且 s[0]==a",
      lambda: tuple(s_1_4.shape) == (2, 2, 3) and close(s_1_4[0], a_1_4) and close(s_1_4[1], b_1_4))
check("1.4c s_last 形状 (2,3,2) 且 s_last[...,0]==a",
      lambda: tuple(s_last_1_4.shape) == (2, 3, 2) and close(s_last_1_4[..., 0], a_1_4) and close(s_last_1_4[..., 1], b_1_4))
check("1.4d 形状表正确",
      lambda: shapes_1_4 == {"c": (4, 3), "s": (2, 2, 3), "s_last": (2, 3, 2)})

### 练习 1.5　广播规则　★

**对应手册**：1.5 矩阵运算 / 广播

预测下面四个加法表达式的结果形状；不能广播的填字符串 `"error"`。答案填进 `broadcast_1_5`：

| 键 | 表达式 |
| --- | --- |
| `a` | `(3, 4) + (4,)` |
| `b` | `(3, 1) + (1, 4)` |
| `c` | `(2, 3, 4) + (3, 1)` |
| `d` | `(5,) + (3, 4)` |

**提示**：从最后一维往左对齐，每一维要么相等，要么其中一方为 1。

In [ ]:
broadcast_1_5 = {
    "a": None,   # TODO: 形状元组，或 "error"
    "b": None,
    "c": None,
    "d": None,
}

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
_pairs_1_5 = [((3, 4), (4,)), ((3, 1), (1, 4)), ((2, 3, 4), (3, 1)), ((5,), (3, 4))]
_names_1_5 = "abcd"

def _expected_1_5(sa, sb):
    try:
        return tuple((torch.zeros(sa) + torch.zeros(sb)).shape)
    except RuntimeError:
        return "error"

for _i, (_sa, _sb) in enumerate(_pairs_1_5):
    check(f"1.5{_names_1_5[_i]}  {_sa} + {_sb}",
          lambda _i=_i, _sa=_sa, _sb=_sb: broadcast_1_5[_names_1_5[_i]] == _expected_1_5(_sa, _sb))

### 练习 1.6　einsum 写注意力分数　★★

**对应手册**：1.5 矩阵运算 / 第 10 章 注意力

`q` 形状 `(2, 4, 5, 8)`，含义是 `(B, H, L, D)`；`k` 形状 `(2, 4, 6, 8)`，含义是 `(B, H, S, D)`。

**要做什么**
1. 用 `torch.einsum` 计算 `scores = q kᵀ`，形状应为 `(B, H, L, S)`，结果赋给 `scores_einsum`。
2. 用 `@` 和 `transpose` 再算一遍，赋给 `scores_matmul`。
3. 判断两者是否 `allclose`，把布尔值赋给 `eq_1_6`。

**提示**：einsum 的下标里，出现两次的维度会被「缩并」（求和），只出现一次的维度保留。

In [ ]:
q_1_6 = torch.randn(2, 4, 5, 8)
k_1_6 = torch.randn(2, 4, 6, 8)

scores_einsum = None  # TODO: torch.einsum("bhld,bhsd->bhls", q_1_6, k_1_6)
scores_matmul = None  # TODO: q_1_6 @ k_1_6.transpose(-1, -2)
eq_1_6 = None         # TODO: 两者是否 allclose，填 True / False

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("1.6a einsum 结果形状 (2,4,5,6)", lambda: tuple(scores_einsum.shape) == (2, 4, 5, 6))
check("1.6b 与 matmul 结果一致", lambda: close(scores_einsum, q_1_6 @ k_1_6.transpose(-1, -2)))
check("1.6c 判断正确", lambda: eq_1_6 is True)

### 练习 1.7　因果 mask　★★

**对应手册**：1.6 归约与索引 / 第 10 章

`scores = torch.randn(4, 4)` 表示 4 个 token 两两的注意力分数。语言模型要求第 `i` 个 token 只能看到 `0..i`，所以要把「未来位置」屏蔽掉。

**要做什么**
1. 构造形状 `(4, 4)` 的 bool 掩码 `mask_1_7`，`True` 表示**屏蔽**（严格上三角，不含对角线）。
2. 用 `masked_fill` 把被屏蔽处填成 `-inf`，得到 `masked_1_7`。
3. 对最后一维做 softmax，得到 `probs_1_7`。

**提示**：`torch.triu(..., diagonal=1)` 正好是严格上三角。

In [ ]:
L_1_7 = 4
scores_1_7 = torch.randn(L_1_7, L_1_7)

mask_1_7 = None     # TODO: (4,4) bool，严格上三角
masked_1_7 = None   # TODO: scores.masked_fill(mask, -inf)
probs_1_7 = None    # TODO: softmax(dim=-1)

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
_tri_1_7 = torch.triu(torch.ones(L_1_7, L_1_7, dtype=torch.bool), diagonal=1)
check("1.7a mask 是严格上三角 bool 张量",
      lambda: mask_1_7.dtype == torch.bool and bool((mask_1_7 == _tri_1_7).all()))
check("1.7b 屏蔽位置为 -inf",
      lambda: bool(torch.isneginf(masked_1_7[mask_1_7]).all()))
check("1.7c 未屏蔽位置数值不变",
      lambda: close(masked_1_7[~mask_1_7], scores_1_7[~mask_1_7]))
check("1.7d 第 0 行概率全部给位置 0",
      lambda: close(probs_1_7[0], torch.tensor([1.0, 0.0, 0.0, 0.0])) and close(probs_1_7.sum(-1), torch.ones(L_1_7)))

### 练习 1.8　视图 / 拷贝 / detach　★★

**对应手册**：2.3 视图 vs 拷贝

```python
b = torch.arange(6.).reshape(2, 3).requires_grad_(True)
v = b.view(3, 2)
c = b.clone()
d = b.detach()
r = b.reshape(3, 2)
```

对这四个结果，判断两件事并填进字典：

1. `shares_1_8`：是否与原张量**共享内存**（用 `data_ptr()` 比较）。
2. `graphs_1_8`：是否**在计算图内**（`requires_grad` 是否为 True）。

**提示**：视图共享内存；`clone` 复制数据但保留计算图；`detach` 共享内存但脱离计算图。`reshape` 在内存连续时返回视图。

In [ ]:
b_1_8 = torch.arange(6.).reshape(2, 3).requires_grad_(True)
v_1_8 = b_1_8.view(3, 2)
c_1_8 = b_1_8.clone()
d_1_8 = b_1_8.detach()
r_1_8 = b_1_8.reshape(3, 2)

shares_1_8 = {"view": None, "clone": None, "detach": None, "reshape": None}
graphs_1_8 = {"view": None, "clone": None, "detach": None, "reshape": None}

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
_objs_1_8 = {"view": v_1_8, "clone": c_1_8, "detach": d_1_8, "reshape": r_1_8}
for _k, _o in _objs_1_8.items():
    check(f"1.8 {_k:<8s} 共享内存={shares_1_8[_k]}",
          lambda _k=_k, _o=_o: shares_1_8[_k] is (bool(_o.data_ptr() == b_1_8.data_ptr())))
for _k, _o in _objs_1_8.items():
    check(f"1.8 {_k:<8s} 在计算图内={graphs_1_8[_k]}",
          lambda _k=_k, _o=_o: graphs_1_8[_k] is bool(_o.requires_grad))

In [ ]:
scoreboard()

## 第二部分　自动求导（手册第 3 章）

核心只有四句话：**叶子张量存梯度、梯度是累加不是覆盖、反向一次配一次前向、推理时关掉计算图**。
建议用时 40–60 分钟。

### 练习 2.1　手算梯度并用 backward 验证　★

**对应手册**：3.1 autograd API

对 `y = (x² + 3x).sum()`，其中 `x = [-1, 0, 2]`：

**要做什么**
1. 手算 `dy/dx`（对每个元素求导，就是 `2x + 3`）。
2. 调用 `backward()`，把 `x.grad` 赋给 `grad_2_1`。

**提示**：`.sum()` 把所有元素加成一个标量，标量才能直接 `backward()`；每个元素的梯度互相独立。

In [ ]:
x_2_1 = torch.tensor([-1.0, 0.0, 2.0], requires_grad=True)
y_2_1 = (x_2_1 ** 2 + 3 * x_2_1).sum()

# TODO: 反向传播，并把 x_2_1.grad 赋给 grad_2_1
grad_2_1 = None

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("2.1 梯度为 [1, 3, 7]", lambda: close(grad_2_1, torch.tensor([1.0, 3.0, 7.0])))

### 练习 2.2　梯度累加与 zero_grad　★

**对应手册**：3.2 计算图 / 3.1 no_grad

梯度是**累加**的，不是覆盖的。

**要做什么**
1. 同一个 `w`，每轮**重新前向**再 `backward()`，连续两轮不清理梯度。`w.grad` 是多少？填 `grad_after_two`。
2. 手动把 `w.grad` 置空，再来一轮前向+反向。`w.grad` 是多少？填 `grad_after_reset`。
3. 对**同一张已经反过向的计算图**再调用一次 `backward()`，用 `try/except` 实测，把结果填进 `twice_backward_ok`（True = 没报错）。

**提示**：每轮 `(w ** 2).backward()` 都会**新建**一张图；而 `y = x ** 2; y.backward(); y.backward()` 是同一张图反向两次。

In [ ]:
w_2_2 = torch.tensor(2.0, requires_grad=True)
for _ in range(2):
    (w_2_2 ** 2).backward()

grad_after_two = None    # TODO: float(w_2_2.grad)

w_2_2.grad = None
(w_2_2 ** 2).backward()
grad_after_reset = None  # TODO: float(w_2_2.grad)

x_2_2 = torch.tensor(1.0, requires_grad=True)
y_2_2 = x_2_2 ** 2
y_2_2.backward()
try:
    y_2_2.backward()
    twice_backward_ok = True
except RuntimeError:
    twice_backward_ok = False

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("2.2a 两轮累加后梯度为 8.0", lambda: abs(grad_after_two - 8.0) < 1e-6)
check("2.2b 清空后单轮梯度为 4.0", lambda: abs(grad_after_reset - 4.0) < 1e-6)
check("2.2c 同一张图第二次 backward 会报错", lambda: twice_backward_ok is False)

### 练习 2.3　no_grad 与 detach　★★

**对应手册**：3.3 三种不建图方式

```python
w = torch.tensor(2.0, requires_grad=True)
y = w * 3
with torch.no_grad():
    z = w * 4
loss = (y.detach() * w).sum()
loss.backward()
```

**要做什么**
1. 判断 `y` 和 `z` 是否在计算图内，分别填 `y_req`、`z_req`（bool）。
2. 在 `no_grad` 里对 `z` 调 `backward()`，把是否抛异常填进 `z_backward_fails`（True = 抛了）。
3. 算出 `w.grad`，填 `grad_2_3`。

**提示**：`y.detach()` 是常数 6；`loss = 6 * w`。

In [ ]:
w_2_3 = torch.tensor(2.0, requires_grad=True)
y_2_3 = w_2_3 * 3
with torch.no_grad():
    z_2_3 = w_2_3 * 4

y_req = None   # TODO: bool
z_req = None   # TODO: bool

z_backward_fails = None  # TODO: 在 no_grad 里对 z_2_3 反向，是否抛异常
try:
    z_2_3.backward()
    z_backward_fails = False
except RuntimeError:
    z_backward_fails = True

loss_2_3 = (y_2_3.detach() * w_2_3).sum()
loss_2_3.backward()
grad_2_3 = None  # TODO: float(w_2_3.grad)

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("2.3a y 在计算图中", lambda: y_req is True)
check("2.3b z 不在计算图中", lambda: z_req is False)
check("2.3c 对 z 反向会失败", lambda: z_backward_fails is True)
check("2.3d detach 后的乘积梯度为 6.0", lambda: abs(grad_2_3 - 6.0) < 1e-6)

### 练习 2.4　非叶子张量与 retain_grad　★★

**对应手册**：3.2 计算图长什么样

```python
a = torch.tensor(2.0, requires_grad=True)
b = a * 3
c = b * 4
c.backward()
```

`b` 是中间结果（非叶子张量）。

**要做什么**
1. 观察 `b.grad` 是否为 `None`，把判断填进 `b_grad_none`（bool）。
2. 另起一组：对 `b2` 调用 `retain_grad()` 之后再反向，把 `b2.grad` 填进 `b2_grad`、`a2.grad` 填进 `a2_grad_2_4`。

**提示**：非叶子张量默认不保存梯度；`retain_grad()` 会要求它保存。

In [ ]:
a_2_4 = torch.tensor(2.0, requires_grad=True)
b_2_4 = a_2_4 * 3
c_2_4 = b_2_4 * 4
c_2_4.backward()
b_grad_none = None   # TODO: bool，b_2_4.grad 是否为 None

a2_2_4 = torch.tensor(2.0, requires_grad=True)
b2_2_4 = a2_2_4 * 3
b2_2_4.retain_grad()
c2_2_4 = b2_2_4 * 4
c2_2_4.backward()
b2_grad = None        # TODO: b2_2_4.grad
a2_grad_2_4 = None    # TODO: a2_2_4.grad

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("2.4a 非叶子 b 默认不存梯度", lambda: b_grad_none is True)
check("2.4b retain_grad 后 b.grad = 4", lambda: close(b2_grad, torch.tensor(4.0)))
check("2.4c 叶子 a2.grad = 12", lambda: close(a2_grad_2_4, torch.tensor(12.0)))

In [ ]:
scoreboard()

## 第三部分　Module 骨架与线性层（手册第 4–5 章）

这一部分的重点：**子模块要注册、buffer 不进优化器、Linear 只改最后一维**。
建议用时 60–90 分钟。

### 练习 3.1　组装 MLP　★

**对应手册**：4.1 核心类 / 5.1 nn.Linear

用 `nn.Sequential` 组装：`Linear(4→16)` → `GELU` → `Dropout(0.1)` → `Linear(16→3)`。

**要做什么**
1. 把模型赋给 `mlp_3_1`。
2. 它应支持任意前导维度：输入 `(2, 5, 4)` 输出 `(2, 5, 3)`。
3. 统计总参数量（含 bias），赋给 `param_count_3_1`。

**提示**：激活函数和 Dropout 没有参数；`Linear(in,out)` 有 `in×out + out` 个参数。

In [ ]:
mlp_3_1 = None          # TODO: 用 nn.Sequential 组装
param_count_3_1 = None  # TODO: sum(p.numel() for p in ...)

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("3.1a 是 nn.Module", lambda: isinstance(mlp_3_1, nn.Module))
check("3.1b 输出形状 (2,5,3)", lambda: tuple(mlp_3_1(torch.randn(2, 5, 4)).shape) == (2, 5, 3))
check("3.1c 参数量 131", lambda: param_count_3_1 == 131 == sum(p.numel() for p in mlp_3_1.parameters()))

### 练习 3.2　参数量手算　★

**对应手册**：5.4 参数量估算

模型结构已在代码里给出。**先手算**总参数量填进 `params_3_2`，再用代码统计填 `computed_3_2`，两者应当相等。

**提示**：两个 `Linear` 各有 weight 和 bias：`10×20+20` 与 `20×5+5`。

In [ ]:
model_3_2 = nn.Sequential(
    nn.Linear(10, 20), nn.ReLU(),
    nn.Linear(20, 5),
)

params_3_2 = None    # TODO: 手算结果
computed_3_2 = None  # TODO: 用 sum(p.numel() ...) 统计

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("3.2a 手算结果 325", lambda: params_3_2 == 325)
check("3.2b 代码统计与手算一致",
      lambda: computed_3_2 == params_3_2 == sum(p.numel() for p in model_3_2.parameters()))

### 练习 3.3　register_buffer　★★

**对应手册**：4.2 Module 的四个必备动作

下面的 `Scaler` 把 `scale` 直接赋成普通张量。普通属性不会被注册：它不在 `state_dict` 里，也不会随 `.to(device)` 搬家。

**要做什么**：把它改成 `register_buffer("scale", torch.tensor(2.0))`。

**自检会验证**：buffer 进入 `state_dict`、不出现在 `parameters()`、forward 数值正确。

In [ ]:
class Scaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = torch.tensor(2.0)   # TODO: 改成 register_buffer
    def forward(self, x):
        return x * self.scale

scaler_3_3 = Scaler()

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("3.3a scale 进入 state_dict", lambda: "scale" in scaler_3_3.state_dict())
check("3.3b scale 不是可训练参数", lambda: len(list(scaler_3_3.parameters())) == 0)
check("3.3c forward 数值正确", lambda: close(scaler_3_3(torch.ones(3)), torch.full((3,), 2.0)))

### 练习 3.4　ModuleList 与普通 list　★★

**对应手册**：4.1 核心类 / 4.2 注册

写一个 `Stack`：3 个 `Linear(4, 4)` 顺序堆叠，子模块必须用 `nn.ModuleList` 注册。

**要做什么**
1. 补全 `__init__` 并实例化，赋给 `stack_3_4`。
2. 把参数**张量个数**（不是元素总数）填进 `modulelist_param_count`。

**思考**：如果换成普通 Python `list`，会发生什么？
**提示**：每个 `Linear(4,4)` 有 weight 与 bias 两个参数张量。

In [ ]:
class Stack(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = None   # TODO: nn.ModuleList([...])
    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x

stack_3_4 = None               # TODO: 实例化
modulelist_param_count = None  # TODO: 参数张量个数

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("3.4a blocks 是 ModuleList", lambda: isinstance(stack_3_4.blocks, nn.ModuleList))
check("3.4b 参数张量 6 个", lambda: modulelist_param_count == 6 == len(list(stack_3_4.parameters())))
check("3.4c forward 形状 (2,4)", lambda: tuple(stack_3_4(torch.randn(2, 4)).shape) == (2, 4))

### 练习 3.5　冻结底座并只优化头部　★★

**对应手册**：4.3 状态管理 / 21.3 常见需求

`Net35` 有 `backbone` 与 `head`。

**要做什么**
1. 冻结 `backbone`：所有参数 `requires_grad=False`。
2. 创建只包含 `head` 参数的 `AdamW`，学习率 `1e-3`，赋给 `opt_3_5`。

**提示**：`p.requires_grad_(False)` 是原地操作；优化器只传 `net_3_5.head.parameters()`。

In [ ]:
class Net35(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Linear(8, 8)
        self.head = nn.Linear(8, 2)
    def forward(self, x):
        return self.head(self.backbone(x))

net_3_5 = Net35()
# TODO 1: 冻结 backbone
opt_3_5 = None   # TODO 2: AdamW，只传 head 的参数

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("3.5a backbone 全部冻结", lambda: not any(p.requires_grad for p in net_3_5.backbone.parameters()))
check("3.5b head 仍可训练", lambda: all(p.requires_grad for p in net_3_5.head.parameters()))
check("3.5c 优化器只包含 2 个参数张量",
      lambda: sum(len(g["params"]) for g in opt_3_5.param_groups) == 2)

### 练习 3.6　LazyLinear 的初始化时机　★★★

**对应手册**：5.3 nn.LazyLinear

`nn.LazyLinear(3)` 的 `in_features` 由第一次 forward 推断。

**要做什么**
1. 判断第一次 forward 之前参数是否处于未初始化状态，填 `before_forward_uninitialized`（bool）。
2. 用 `(5, 4)` 的输入跑一次，把 `weight.shape` 填进 `after_weight_shape`。

**提示**：用 `torch.nn.parameter.UninitializedParameter` 判断。

In [ ]:
from torch.nn.parameter import UninitializedParameter

lazy_3_6 = nn.LazyLinear(3)
before_forward_uninitialized = None   # TODO: bool

_ = lazy_3_6(torch.randn(5, 4))
after_weight_shape = None             # TODO: tuple(lazy_3_6.weight.shape)

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("3.6a forward 前未初始化", lambda: before_forward_uninitialized is True)
check("3.6b 初始化后 weight 形状 (3,4)", lambda: tuple(after_weight_shape) == (3, 4))

In [ ]:
scoreboard()

## 第四部分　卷积与池化（手册第 6 章）

两块硬功夫：**输出尺寸公式**与**参数量公式**；再加一个「用梯度实测感受野」的技巧。
建议用时 50–70 分钟。

### 练习 4.1　卷积输出尺寸　★★

**对应手册**：6.2 输出尺寸与参数量

输入高 `H = 32`，预测 `Conv2d` 输出的高（整数）：

```
H_out = floor((H + 2p - d(k-1) - 1) / s + 1)
```

| 键 | 配置 |
| --- | --- |
| `a` | k=3, p=1, s=1 |
| `b` | k=3, p=0, s=2 |
| `c` | k=3, p=1, s=2 |
| `d` | k=3, p=0, s=1, dilation=2 |

把四个结果填进 `out_heights_4_1`（值写整数）。

In [ ]:
out_heights_4_1 = {
    "a": None,   # TODO
    "b": None,
    "c": None,
    "d": None,
}

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
_configs_4_1 = {
    "a": dict(kernel_size=3, padding=1, stride=1, dilation=1),
    "b": dict(kernel_size=3, padding=0, stride=2, dilation=1),
    "c": dict(kernel_size=3, padding=1, stride=2, dilation=1),
    "d": dict(kernel_size=3, padding=0, stride=1, dilation=2),
}
for _k, _cfg in _configs_4_1.items():
    def _run_4_1(_k=_k, _cfg=_cfg):
        y = nn.Conv2d(1, 1, **_cfg)(torch.zeros(1, 1, 32, 32))
        return out_heights_4_1[_k] == y.shape[-1]
    check(f"4.1{_k} 输出高", _run_4_1)

### 练习 4.2　卷积参数量与 groups　★★

**对应手册**：6.2 参数量 / 深度可分离卷积

参数量公式（`bias=True`）：

```
out_channels × (in_channels / groups) × k × k + out_channels
```

| 键 | 配置 |
| --- | --- |
| `a` | `Conv2d(3, 8, 3, groups=1)` |
| `b` | `Conv2d(3, 6, 3, groups=3)` |
| `c` | `Conv2d(3, 3, 3, groups=3)`（depthwise） |

把三个参数量填进 `conv_params_4_2`。**注意**：`groups` 必须同时整除输入与输出通道数。

In [ ]:
conv_params_4_2 = {"a": None, "b": None, "c": None}  # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
_layers_4_2 = {
    "a": nn.Conv2d(3, 8, 3, groups=1),
    "b": nn.Conv2d(3, 6, 3, groups=3),
    "c": nn.Conv2d(3, 3, 3, groups=3),
}
for _k, _layer in _layers_4_2.items():
    check(f"4.2{_k} 参数量",
          lambda _k=_k, _layer=_layer: conv_params_4_2[_k] == sum(p.numel() for p in _layer.parameters()))

### 练习 4.3　CNN 分类头与全局平均池化　★★

**对应手册**：6.3 池化家族 / 附录 A.2

补全 `SmallCNN` 的分类头 `Linear(16, num_classes)`，并回答两个形状/数量问题。

结构：`Conv2d(3,8,3,padding=1)` → ReLU → `MaxPool2d(2)` → `Conv2d(8,16,3,padding=1)` → ReLU → `AdaptiveAvgPool2d(1)` → flatten → head。输入 `(2, 3, 32, 32)`。

**要做什么**
1. 补全 `self.head`。
2. 填 GAP 之后 `features` 输出的形状 `pooled_shape_4_3`。
3. 填总参数量 `param_count_4_3`。

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = None   # TODO: nn.Linear(16, num_classes)
    def forward(self, x):
        return self.head(self.features(x).flatten(1))

cnn_4_3 = SmallCNN()
pooled_shape_4_3 = None   # TODO: features 输出的形状
param_count_4_3 = None    # TODO: 总参数量

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("4.3a 输出 (2,10)",
      lambda: tuple(cnn_4_3(torch.randn(2, 3, 32, 32)).shape) == (2, 10))
check("4.3b GAP 后形状 (2,16,1,1)", lambda: tuple(pooled_shape_4_3) == (2, 16, 1, 1))
check("4.3c 参数量 1562", lambda: param_count_4_3 == 1562 == sum(p.numel() for p in cnn_4_3.parameters()))

### 练习 4.4　用梯度实测感受野　★★★

**对应手册**：6.2 感受野 / 平移等变

三个**无 padding、stride=1** 的 3×3 卷积串联，输入 9×9。对中间输出 `y[0,0,1,1]` 反向，观察输入梯度的非零区域。

**要做什么**
1. 手算理论感受野边长，填 `rf_4_4`。
2. 实测输入梯度中非零元素的个数，填 `measured_4_4`。

**提示**：每经过一层 3×3 卷积，感受野边长 +2。

In [ ]:
net_4_4 = nn.Sequential(
    nn.Conv2d(1, 1, 3), nn.Conv2d(1, 1, 3), nn.Conv2d(1, 1, 3),
)
x_4_4 = torch.randn(1, 1, 9, 9, requires_grad=True)
y_4_4 = net_4_4(x_4_4)
y_4_4[0, 0, 1, 1].backward()

rf_4_4 = None        # TODO: 理论感受野边长
measured_4_4 = None  # TODO: x_4_4.grad 中非零元素个数

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("4.4a 理论感受野 = 7", lambda: rf_4_4 == 7)
check("4.4b 实测面积 = 49", lambda: measured_4_4 == 49)
check("4.4c 实测区域为 7×7",
      lambda: int((x_4_4.grad[0, 0].abs() > 0).sum(dim=1).max()) == 7)

In [ ]:
scoreboard()

## 第五部分　循环网络（手册第 7 章）

核心是形状：**默认 `(L, N, H)`，`batch_first=True` 才是 `(N, L, H)`**；变长序列要 `pad` + `pack`。
建议用时 50–70 分钟。

### 练习 5.1　batch_first 与 LSTM 输出形状　★

**对应手册**：7.1 类与形状

`L=10, N=2, input_size=16, hidden_size=32`。

**要做什么**
1. 默认 LSTM（`batch_first=False`）吃 `(10, 2, 16)`，填 `output` 与 `h_n` 形状。
2. `batch_first=True` 的 LSTM 吃 `(2, 10, 16)`，填对应形状。
3. 填 `lstm_default_batch_first`（默认值，bool）。

In [ ]:
lstm_5_1 = nn.LSTM(16, 32)
out_5_1, (h_5_1, c_5_1) = lstm_5_1(torch.randn(10, 2, 16))

lstm_bf_5_1 = nn.LSTM(16, 32, batch_first=True)
out_bf_5_1, (h_bf_5_1, c_bf_5_1) = lstm_bf_5_1(torch.randn(2, 10, 16))

shapes_5_1 = {"out": None, "h": None, "out_bf": None, "h_bf": None}  # TODO
lstm_default_batch_first = None  # TODO: True / False

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("5.1a 默认 out (10,2,32)", lambda: shapes_5_1["out"] == (10, 2, 32) == tuple(out_5_1.shape))
check("5.1b 默认 h_n (1,2,32)", lambda: shapes_5_1["h"] == (1, 2, 32) == tuple(h_5_1.shape))
check("5.1c batch_first out (2,10,32)", lambda: shapes_5_1["out_bf"] == (2, 10, 32) == tuple(out_bf_5_1.shape))
check("5.1d batch_first h_n (1,2,32)", lambda: shapes_5_1["h_bf"] == (1, 2, 32) == tuple(h_bf_5_1.shape))
check("5.1e 默认 batch_first 为 False",
      lambda: lstm_default_batch_first is False and lstm_5_1.batch_first is False)

### 练习 5.2　pad_sequence 与 pack_padded_sequence　★★

**对应手册**：7.3 配套工具 / 15.4 collate

三条变长序列，长度为 3、1、2。

**要做什么**
1. 用 `pad_sequence(seqs, batch_first=True)` 补齐，得到 `padded_5_2`。
2. 构造 `lengths_5_2 = tensor([3,1,2])`。
3. 用 `pack_padded_sequence`（`enforce_sorted=False`）打包后喂给 `GRU(1→8, batch_first=True)`，把 `packed_out.data` 的形状填进 `packed_data_shape_5_2`。

**提示**：pad_sequence 得到 `(3, 3)`，喂 GRU 前要 `unsqueeze(-1)` 变成 `(3, 3, 1)`。

In [ ]:
seqs_5_2 = [torch.tensor([1.0, 2.0, 3.0]), torch.tensor([4.0]), torch.tensor([5.0, 6.0])]

padded_5_2 = None    # TODO
lengths_5_2 = None   # TODO
packed_5_2 = None    # TODO

gru_5_2 = nn.GRU(1, 8, batch_first=True)
packed_out_5_2 = None
packed_data_shape_5_2 = None  # TODO: tuple(packed_out_5_2.data.shape)

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("5.2a padded 形状 (3,3)", lambda: tuple(padded_5_2.shape) == (3, 3))
check("5.2b 补齐位置为 0", lambda: float(padded_5_2[1, 1:].abs().sum()) == 0.0)
check("5.2c lengths = [3,1,2]", lambda: close(lengths_5_2, torch.tensor([3, 1, 2])))
check("5.2d packed data 形状 (6,8)", lambda: tuple(packed_data_shape_5_2) == (6, 8))

### 练习 5.3　GRU 文本分类器　★★

**对应手册**：7.2 三个循环单元 / 附录 A.3

补全 `TextClf`：`Embedding(50, 16, padding_idx=0)` → `GRU(16→32, batch_first=True)` → 取最后一个时间步 → `Linear(32, 2)`。

**要做什么**
1. 补全 `self.rnn` 与 `self.head`。
2. 输入 ids 形状 `(2, 7)`，把输出 logits 的形状填进 `logits_shape_5_3`。
3. 填总参数量 `param_count_5_3`。

In [ ]:
class TextClf(nn.Module):
    def __init__(self, vocab=50, d=16, hidden=32, classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=0)
        self.rnn = None   # TODO: nn.GRU(...)
        self.head = None  # TODO: nn.Linear(...)
    def forward(self, ids):
        out, _ = self.rnn(self.emb(ids))
        return self.head(out[:, -1])

clf_5_3 = TextClf()
logits_shape_5_3 = None   # TODO
param_count_5_3 = None    # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("5.3a logits 形状 (2,2)",
      lambda: logits_shape_5_3 == (2, 2)
      == tuple(clf_5_3(torch.tensor([[1, 2, 3], [4, 5, 6]])).shape))
check("5.3b 参数量 5666", lambda: param_count_5_3 == 5666 == sum(p.numel() for p in clf_5_3.parameters()))

### 练习 5.4　手写 LSTM 单步　★★★

**对应手册**：7.2 LSTM 的门

用公式手写一个 LSTM 单步，与 `nn.LSTMCell` 对齐：

```
i = σ(W_i x + U_i h + b_i)      输入门
f = σ(W_f x + U_f h + b_f)      遗忘门
g = tanh(W_g x + U_g h + b_g)   候选值
o = σ(W_o x + U_o h + b_o)      输出门
c' = f ⊙ c + i ⊙ g
h' = o ⊙ tanh(c')
```

**注意**：PyTorch 把四个门拼在 `weight_ih` / `weight_hh` 里，顺序是 **i, f, g, o**。

**要做什么**：计算 `gates_5_4`，切出 `i/f/g/o`，再算出 `c_5_4` 与 `h_5_4`；它们应与 `nn.LSTMCell` 的输出在 1e-6 内一致。

In [ ]:
torch.manual_seed(1)
cell_5_4 = nn.LSTMCell(8, 16)
x_5_4 = torch.randn(2, 8)
h0_5_4 = torch.randn(2, 16)
c0_5_4 = torch.randn(2, 16)
h_ref_5_4, c_ref_5_4 = cell_5_4(x_5_4, (h0_5_4, c0_5_4))

gates_5_4 = None
i_5_4 = f_5_4 = g_5_4 = o_5_4 = None
c_5_4 = h_5_4 = None
# TODO: 按公式计算（门的顺序 i, f, g, o）

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("5.4a 四个门形状 (2,16)",
      lambda: all(tuple(t.shape) == (2, 16) for t in (i_5_4, f_5_4, g_5_4, o_5_4)))
check("5.4b h 与 nn.LSTMCell 一致", lambda: close(h_5_4, h_ref_5_4, atol=1e-6))
check("5.4c c 与 nn.LSTMCell 一致", lambda: close(c_5_4, c_ref_5_4, atol=1e-6))

In [ ]:
scoreboard()

## 第六部分　归一化、激活与 Dropout（手册第 8–9 章）

归一化记统计范围（batch / 样本 / 通道组），Dropout 记「训练缩放、推理直通」。
这一部分大量是「手写一遍再对比官方实现」，比背公式有效。
建议用时 60–80 分钟。

### 练习 6.1　手写 LayerNorm　★★

**对应手册**：8.2 公式对照

`nn.LayerNorm(6)` 对**最后一个维度**做：
`(x - μ) / sqrt(σ² + ε) * γ + β`，其中 μ、σ² 是每个样本在这 6 维上的均值与方差，`ε = 1e-5`。

**要做什么**：手写 `y_manual_6_1`，与 `ln_6_1(x)` 在 1e-5 内一致。

**提示**：`x.var(dim=-1, unbiased=False, keepdim=True)` 才是总体方差。

In [ ]:
torch.manual_seed(13)
x_6_1 = torch.randn(4, 6)
ln_6_1 = nn.LayerNorm(6)
y_ref_6_1 = ln_6_1(x_6_1)

y_manual_6_1 = None   # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("6.1 手写结果与 nn.LayerNorm 一致", lambda: close(y_manual_6_1, y_ref_6_1, atol=1e-5))

### 练习 6.2　手写 RMSNorm　★★

**对应手册**：8.2 RMSNorm

RMSNorm 不减均值、没有 β：

```
y = x / sqrt(mean(x²) + ε) * γ
```

**要做什么**：手写 `y_manual_6_2`，与 `nn.RMSNorm(6, eps=1e-6)` 的输出在 1e-5 内一致。

**提示**：均值只对最后一维求，`keepdim=True`。

In [ ]:
torch.manual_seed(14)
x_6_2 = torch.randn(4, 6)
rms_6_2 = nn.RMSNorm(6, eps=1e-6)
y_ref_6_2 = rms_6_2(x_6_2)

y_manual_6_2 = None   # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("6.2 手写结果与 nn.RMSNorm 一致", lambda: close(y_manual_6_2, y_ref_6_2, atol=1e-5))

### 练习 6.3　BatchNorm 的训练与推理　★★

**对应手册**：8.3 训练/推理的差别

**要做什么**
1. 在 train 模式用 `(8, 3)` 的批前向一次，判断 `running_mean` 是否变化，填 `running_changed_6_3`。
2. 切到 eval，对 `x_eval_6_3` 前向。用 `running_mean` / `running_var` 手算期望输出，填 `expected_eval_6_3`。
3. 判断 eval 是否确实使用滑动统计，填 `eval_uses_running_6_3`。
4. train 模式下 batch=1 前向是否抛 `ValueError`？填 `batch1_raises`。

**提示**：eval 的公式仍是 `(x - running_mean) / sqrt(running_var + ε) * γ + β`。

In [ ]:
torch.manual_seed(2)
bn_6_3 = nn.BatchNorm1d(3)
before_rm_6_3 = bn_6_3.running_mean.clone()
bn_6_3.train()
_ = bn_6_3(torch.randn(8, 3) * 2 + 5)

running_changed_6_3 = None   # TODO: bool

bn_6_3.eval()
x_eval_6_3 = torch.randn(8, 3) * 10 + 100
out_eval_6_3 = bn_6_3(x_eval_6_3)
expected_eval_6_3 = None      # TODO: 用 running 统计手算
eval_uses_running_6_3 = None  # TODO: bool

bn_6_3.train()
try:
    bn_6_3(torch.randn(1, 3))
    batch1_raises = False
except ValueError:
    batch1_raises = True

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("6.3a 训练模式更新了 running_mean",
      lambda: running_changed_6_3 is True and float(bn_6_3.running_mean.abs().sum()) > 0.0)
check("6.3b eval 用滑动统计（手算一致）",
      lambda: expected_eval_6_3 is not None and close(out_eval_6_3, expected_eval_6_3, atol=1e-5))
check("6.3c 你的判断正确", lambda: eval_uses_running_6_3 is True)
check("6.3d batch=1 训练报 ValueError", lambda: batch1_raises is True)

### 练习 6.4　Dropout 的期望保持　★★

**对应手册**：9.2 Dropout 家族

`nn.Dropout(0.5)`，输入是一万个 1。

**要做什么**（训练模式下）
1. 存活元素的值是多少？填 `drop_scale_6_4`。
2. 被置零的比例是多少？填 `zero_frac_6_4`。
3. 输出均值是否仍接近 1？填 `mean_6_4`（bool）。
4. 切到 eval 后输出是否与输入完全相同？填 `eval_identity_6_4`。

In [ ]:
torch.manual_seed(3)
drop_6_4 = nn.Dropout(0.5)
drop_6_4.train()
x_6_4 = torch.ones(100_000)
y_6_4 = drop_6_4(x_6_4)

drop_scale_6_4 = None    # TODO: 存活元素的值
zero_frac_6_4 = None     # TODO: 置零比例（float）
mean_6_4 = None          # TODO: 均值是否接近 1（bool）

drop_6_4.eval()
y_eval_6_4 = drop_6_4(x_6_4)
eval_identity_6_4 = None # TODO: bool

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("6.4a 存活元素为 2.0", lambda: abs(drop_scale_6_4 - 2.0) < 1e-6)
check("6.4b 置零比例 ≈ 0.5", lambda: abs(zero_frac_6_4 - 0.5) < 0.02)
check("6.4c 训练模式期望不变", lambda: mean_6_4 is True)
check("6.4d eval 原样输出", lambda: eval_identity_6_4 is True and close(y_eval_6_4, x_6_4))

### 练习 6.5　手写 GELU　★★

**对应手册**：9.1 激活函数全家福

精确 GELU 的定义：

```
GELU(x) = x · Φ(x) = x · 0.5 · (1 + erf(x / √2))
```

**要做什么**：手写 `gelu_manual_6_5`，与 `F.gelu(x)` 在 1e-6 内一致。

**提示**：`torch.erf` 就是误差函数；`math.sqrt(2)` 即可。

In [ ]:
x_6_5 = torch.linspace(-4, 4, 17)
gelu_ref_6_5 = F.gelu(x_6_5)

gelu_manual_6_5 = None   # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("6.5 手写 GELU 与 F.gelu 一致", lambda: close(gelu_manual_6_5, gelu_ref_6_5, atol=1e-6))

### 练习 6.6　激活与模式选择（选择题）　★

**对应手册**：第 8–9 章 综合

把每题的选项字母填进对应变量（`answer_6_6a` … `answer_6_6d`）。

**a)** 多分类网络输出层要得到概率分布，应接：
A. `Sigmoid`　B. `Softmax`　C. `GELU`　D. `Tanh`

**b)** Transformer 的 FFN 最常用：
A. `Sigmoid`　B. `ReLU`　C. `GELU`　D. `Softplus`

**c)** 训练一段时间后大量神经元输出恒为 0，最可能的原因是：
A. tanh 饱和　B. sigmoid 饱和　C. ReLU 死亡　D. Dropout 没关

**d)** 推理结果每次都不一样，最可能的原因是：
A. 学习率太大　B. 忘了 `model.eval()`　C. 没有 warmup　D. 没有梯度裁剪

In [ ]:
answer_6_6a = None
answer_6_6b = None
answer_6_6c = None
answer_6_6d = None

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
quiz("6.6a 多分类输出层", answer_6_6a, "B", "多分类用 softmax 归一化成概率；sigmoid 用于二分类/多标签。")
quiz("6.6b Transformer FFN", answer_6_6b, "C", "BERT/GPT 系默认 GELU。")
quiz("6.6c 神经元恒为 0", answer_6_6c, "C", "ReLU 在负半轴梯度为 0，学习率过大会让神经元永久死亡。")
quiz("6.6d 推理结果抖动", answer_6_6d, "B", "eval 模式才会关闭 Dropout 并用 BN 的滑动统计。")

In [ ]:
scoreboard()

## 第七部分　Embedding 与注意力（手册第 10 章）

这一部分是全书离大模型最近的一块：查表、缩放点积、多头拆分、RoPE。
建议用时 70–100 分钟。

### 练习 7.1　Embedding 就是查表　★

**对应手册**：10.1 nn.Embedding

`emb = nn.Embedding(10, 4, padding_idx=0)`，`ids = [[1,2,0],[3,0,0]]`。

**要做什么**
1. 填输出形状 `emb_out_shape_7_1`。
2. `padding_idx` 对应的权重行是否为全 0？填 `pad_zero_7_1`（bool）。
3. 用 `F.one_hot(ids, 10).float() @ emb.weight` 验证与查表等价，结果赋给 `onehot_out_7_1`。

In [ ]:
emb_7_1 = nn.Embedding(10, 4, padding_idx=0)
ids_7_1 = torch.tensor([[1, 2, 0], [3, 0, 0]])
out_7_1 = emb_7_1(ids_7_1)

emb_out_shape_7_1 = None  # TODO
pad_zero_7_1 = None       # TODO: bool
onehot_out_7_1 = None     # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("7.1a 输出形状 (2,3,4)", lambda: tuple(out_7_1.shape) == (2, 3, 4) == emb_out_shape_7_1)
check("7.1b padding_idx 行为全 0",
      lambda: pad_zero_7_1 is True and float(emb_7_1.weight[0].abs().sum()) == 0.0)
check("7.1c one-hot 乘法等价", lambda: close(onehot_out_7_1, out_7_1))

### 练习 7.2　手写缩放点积注意力　★★★

**对应手册**：10.2 注意力：从公式到 API

实现 `sdpa_manual(q, k, v, attn_mask=None)`：

```
scores = q kᵀ / sqrt(d_k)
scores = scores.masked_fill(attn_mask, -inf)   # attn_mask 为 bool，True = 屏蔽
return softmax(scores, -1) v
```

**注意语义**：`F.scaled_dot_product_attention` 的 **bool** `attn_mask` 恰好相反（True = 参与计算），所以本练习只用它做「无 mask」与 `is_causal=True` 的对比。

**要做什么**
1. 完成函数。
2. 算出无 mask 的 `out_plain_7_2`。
3. 用严格上三角 mask 算出 `out_causal_7_2`，它应与 `is_causal=True` 的结果一致。

In [ ]:
def sdpa_manual(q, k, v, attn_mask=None):
    # TODO: 缩放点积注意力；attn_mask 为 bool，True = 屏蔽
    raise NotImplementedError

q_7_2 = torch.randn(2, 2, 5, 8)
k_7_2 = torch.randn(2, 2, 5, 8)
v_7_2 = torch.randn(2, 2, 5, 8)
mask_7_2 = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)

out_plain_7_2 = None    # TODO
out_causal_7_2 = None   # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("7.2a 无 mask 与 F.sdpa 一致",
      lambda: close(sdpa_manual(q_7_2, k_7_2, v_7_2),
                    F.scaled_dot_product_attention(q_7_2, k_7_2, v_7_2), atol=1e-5))
check("7.2b 因果 mask 与 is_causal 一致",
      lambda: close(sdpa_manual(q_7_2, k_7_2, v_7_2, mask_7_2),
                    F.scaled_dot_product_attention(q_7_2, k_7_2, v_7_2, is_causal=True), atol=1e-5))
check("7.2c out_plain 正确",
      lambda: close(out_plain_7_2, F.scaled_dot_product_attention(q_7_2, k_7_2, v_7_2), atol=1e-5))
check("7.2d out_causal 正确",
      lambda: close(out_causal_7_2, F.scaled_dot_product_attention(q_7_2, k_7_2, v_7_2, is_causal=True), atol=1e-5))

### 练习 7.3　多头拆分与合并　★★

**对应手册**：10.2 多头为什么有用

`x` 形状 `(2, 5, 8)`（B, L, E），头数 `H = 2`。

**要做什么**
1. 拆成 `(B, L, H, d)`：`x_split_7_3`。
2. 换轴成 `(B, H, L, d)`：`x_heads_7_3`（用 `permute`）。
3. 合并回 `(B, L, E)`：`x_merged_7_3`，数值应与原张量一致。
4. 填 `shapes_7_3`。

In [ ]:
x_7_3 = torch.randn(2, 5, 8)
B_7_3, L_7_3, E_7_3 = x_7_3.shape
H_7_3 = 2

x_split_7_3 = None    # TODO
x_heads_7_3 = None    # TODO
x_merged_7_3 = None   # TODO
shapes_7_3 = {"split": None, "heads": None, "merged": None}

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("7.3a split (2,5,2,4)", lambda: tuple(x_split_7_3.shape) == (2, 5, 2, 4))
check("7.3b heads (2,2,5,4)", lambda: tuple(x_heads_7_3.shape) == (2, 2, 5, 4))
check("7.3c merged 还原且数值一致",
      lambda: tuple(x_merged_7_3.shape) == (2, 5, 8) and close(x_merged_7_3, x_7_3))
check("7.3d 分头取值正确",
      lambda: close(x_heads_7_3[0, 0, 0], x_7_3[0, 0, :4]) and close(x_heads_7_3[0, 1, 0], x_7_3[0, 0, 4:]))
check("7.3e 形状表正确",
      lambda: shapes_7_3 == {"split": (2, 5, 2, 4), "heads": (2, 2, 5, 4), "merged": (2, 5, 8)})

### 练习 7.4　MultiheadAttention 的形状　★★

**对应手册**：10.2 MultiheadAttention

`nn.MultiheadAttention(embed_dim=8, num_heads=2)`，默认 `batch_first=False`。

**要做什么**
1. `q=(5,3,8)`，`k=v=(6,3,8)`，填输出 `out` 与注意力权重 `weights` 的形状到 `shapes_7_4`。
2. 再用 `batch_first=True` 的版本，输入 `(3,5,8)` / `(3,6,8)`，填 `shapes_bf_7_4`。

**提示**：weights 的布局是 `(N, L, S)`，默认已对各头取平均。

In [ ]:
mha_7_4 = nn.MultiheadAttention(8, 2)
q_7_4 = torch.randn(5, 3, 8)
kv_7_4 = torch.randn(6, 3, 8)
out_7_4, w_7_4 = mha_7_4(q_7_4, kv_7_4, kv_7_4)
shapes_7_4 = {"out": None, "weights": None}   # TODO

mha_bf_7_4 = nn.MultiheadAttention(8, 2, batch_first=True)
out_bf_7_4, w_bf_7_4 = mha_bf_7_4(
    torch.randn(3, 5, 8), torch.randn(3, 6, 8), torch.randn(3, 6, 8))
shapes_bf_7_4 = {"out": None, "weights": None}  # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("7.4a 默认 out (5,3,8)",
      lambda: shapes_7_4["out"] == (5, 3, 8) == tuple(out_7_4.shape))
check("7.4b weights (3,5,6)",
      lambda: shapes_7_4["weights"] == (3, 5, 6) == tuple(w_7_4.shape))
check("7.4c batch_first out (3,5,8)",
      lambda: shapes_bf_7_4["out"] == (3, 5, 8) == tuple(out_bf_7_4.shape))
check("7.4d batch_first weights (3,5,6)",
      lambda: shapes_bf_7_4["weights"] == (3, 5, 6) == tuple(w_bf_7_4.shape))

### 练习 7.5　简化版 RoPE　★★★

**对应手册**：10.4 位置编码

实现简化版旋转位置编码（RoPE）：

1. `precompute_freqs(dim, max_len, theta=10000.0)`：
   `freqs = 1 / theta^(2i/dim)`（i=0..dim/2-1），`angles = t ⊗ freqs`，返回 `(cos, sin)`，形状都是 `(max_len, dim/2)`。
2. `apply_rope(x, cos, sin)`：把 x 最后一维前后两半看作 `(x1, x2)`，做二维旋转：
   `[x1·cos − x2·sin, x1·sin + x2·cos]`。
3. 用同一个固定向量 `a`、`b` 在每个位置旋转，验证 `q_m·k_n` 只依赖相对位置 `m−n`（即 `D[i,j] == D[i+1,j+1]`）。

**要做什么**：补全两个函数，并填 `qa_7_5`、`kb_7_5`、`relative_ok_7_5`。

In [ ]:
def precompute_freqs(dim, max_len, theta=10000.0):
    # TODO
    raise NotImplementedError

def apply_rope(x, cos, sin):
    # TODO
    raise NotImplementedError

cos_7_5 = sin_7_5 = None      # TODO: precompute_freqs(8, 16)
x_rope_7_5 = torch.randn(16, 8)
q_rope_7_5 = None             # TODO: apply_rope(x_rope_7_5, cos, sin)

a_7_5 = torch.randn(8)
b_7_5 = torch.randn(8)
qa_7_5 = None   # TODO: apply_rope(a.expand(16, 8), cos, sin)
kb_7_5 = None   # TODO: apply_rope(b.expand(16, 8), cos, sin)
relative_ok_7_5 = None   # TODO: D[i,j] == D[i+1,j+1] 是否成立

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("7.5a cos/sin 形状 (16,4)",
      lambda: tuple(cos_7_5.shape) == (16, 4) and tuple(sin_7_5.shape) == (16, 4))
check("7.5b 旋转保持范数",
      lambda: close(q_rope_7_5.norm(dim=-1), x_rope_7_5.norm(dim=-1), atol=1e-5))
check("7.5c 位置 0 不旋转",
      lambda: close(q_rope_7_5[0], x_rope_7_5[0], atol=1e-6))
check("7.5d 相对位置性质", lambda: relative_ok_7_5 is True)

In [ ]:
scoreboard()

## 第八部分　损失函数（手册第 11 章）

记住三条：**CE 吃 logits、target 是 long、不要手动 softmax**。
建议用时 50–70 分钟。

### 练习 8.1　交叉熵 = NLL(log_softmax)　★

**对应手册**：11.1 分类损失

对同一组 logits 与 target，用三种方式算损失，结果应一致：

1. `nn.CrossEntropyLoss()`。
2. `nn.NLLLoss()` + `F.log_softmax`。
3. 手动：`-log_softmax(logits)[range, target].mean()`。

In [ ]:
torch.manual_seed(5)
logits_8_1 = torch.randn(4, 3)
target_8_1 = torch.tensor([0, 2, 1, 1])

loss_ce_8_1 = None      # TODO
loss_nll_8_1 = None     # TODO
loss_manual_8_1 = None  # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("8.1a CE 与 NLL 一致", lambda: close(loss_ce_8_1, loss_nll_8_1, atol=1e-6))
check("8.1b CE 与手算一致", lambda: close(loss_ce_8_1, loss_manual_8_1, atol=1e-6))
check("8.1c 损失是标量", lambda: loss_ce_8_1.dim() == 0)

### 练习 8.2　标签平滑手算　★★

**对应手册**：11.1 CrossEntropyLoss 参数

标签平滑把 one-hot 目标改成 `(1−ε)·one-hot + ε/C`，交叉熵变成：

```
loss = -[(1−ε)·log p_y + (ε/C)·Σ_j log p_j]   （再对 batch 取平均）
```

**要做什么**
1. `loss_ls_8_2` 用 `nn.CrossEntropyLoss(label_smoothing=0.1)` 计算。
2. `loss_manual_8_2` 按上式手算。
两者应在 1e-6 内一致。

In [ ]:
torch.manual_seed(5)
logits_8_2 = torch.randn(4, 3)
target_8_2 = torch.tensor([0, 2, 1, 1])

eps_8_2 = 0.1
loss_ls_8_2 = nn.CrossEntropyLoss(label_smoothing=eps_8_2)(logits_8_2, target_8_2)
logp_8_2 = F.log_softmax(logits_8_2, dim=-1)
loss_manual_8_2 = None   # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("8.2 标签平滑与手算一致", lambda: close(loss_ls_8_2, loss_manual_8_2, atol=1e-6))

### 练习 8.3　ignore_index 与全忽略的 nan　★★

**对应手册**：11.1 / 11.4 坑

`target = [0, 1, -100]`，`ignore_index=-100`。

**要做什么**
1. 用 `nn.CrossEntropyLoss(ignore_index=-100)` 算损失，填 `loss_ii_8_3`。
2. 只对前两条算普通 CE，填 `loss_manual_8_3`，两者应一致。
3. 把三条 target 全部设成 -100，算出的损失填 `loss_all_ignored_8_3`。
4. 这个损失是否为 nan？填 `all_ignored_is_nan_8_3`。

In [ ]:
logits_8_3 = torch.randn(3, 3)
target_8_3 = torch.tensor([0, 1, -100])

loss_ii_8_3 = nn.CrossEntropyLoss(ignore_index=-100)(logits_8_3, target_8_3)
loss_manual_8_3 = None          # TODO
loss_all_ignored_8_3 = None     # TODO
all_ignored_is_nan_8_3 = None   # TODO: bool

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("8.3a ignore_index 与只算前两条一致",
      lambda: close(loss_ii_8_3, loss_manual_8_3, atol=1e-6))
check("8.3b 全部忽略时损失为 nan", lambda: bool(torch.isnan(loss_all_ignored_8_3)))
check("8.3c 判断正确", lambda: all_ignored_is_nan_8_3 is True)

### 练习 8.4　BCEWithLogits 与 BCELoss　★

**对应手册**：11.1 分类损失

对同一组 logits 与 0/1 目标：

1. `loss_bce_logits_8_4` 用 `nn.BCEWithLogitsLoss()`。
2. `loss_bce_8_4` 用 `nn.BCELoss()`，但输入必须先 `sigmoid`。

In [ ]:
torch.manual_seed(6)
logits_8_4 = torch.randn(6)
target_8_4 = torch.tensor([0.0, 1.0, 0.0, 1.0, 1.0, 0.0])

loss_bce_logits_8_4 = None  # TODO
loss_bce_8_4 = None         # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("8.4 两种 BCE 结果一致", lambda: close(loss_bce_logits_8_4, loss_bce_8_4, atol=1e-6))

### 练习 8.5　KLDivLoss 的 reduction　★★

**对应手册**：11.3 分布/度量损失

KLDivLoss 的输入是 **log 概率**，目标默认是概率。两种 reduction 差别很大：

- `reduction='mean'`：除以元素总数（N×C）。
- `reduction='batchmean'`：除以 batch 大小 N。

**要做什么**
1. `kl_batchmean_8_5` 用 `reduction='batchmean'`，注意 `KLDivLoss(input=log q, target=p)` 的 target 是**概率**。
2. `kl_mean_8_5` 用 `reduction='mean'`。
3. `kl_manual_8_5` 手算 `Σ p·(log p − log q)` 再对 batch 取平均。

**提示**：`p` 的概率就是 `logp.exp()`；target 传 log 概率会得到 nan。

In [ ]:
torch.manual_seed(15)
p_logits_8_5 = torch.randn(4, 5)
q_logits_8_5 = torch.randn(4, 5)
logp_8_5 = F.log_softmax(p_logits_8_5, dim=-1)
logq_8_5 = F.log_softmax(q_logits_8_5, dim=-1)

p_prob_8_5 = None       # TODO: logp 的指数
kl_batchmean_8_5 = None  # TODO
kl_mean_8_5 = None       # TODO
kl_manual_8_5 = None     # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("8.5a batchmean 与手算一致", lambda: close(kl_batchmean_8_5, kl_manual_8_5, atol=1e-6))
check("8.5b mean 是按元素平均（batchmean / C）",
      lambda: close(kl_mean_8_5, kl_manual_8_5 / 5, atol=1e-6))

In [ ]:
scoreboard()

## 第九部分　优化器与学习率（手册第 12–13 章）

四件事：**SGD 的更新式、Adam/AdamW 的衰减差别、调度器的顺序、参数组**。
建议用时 50–70 分钟。

### 练习 9.1　SGD 走一步　★

**对应手册**：12.1 优化器全家福

`w = 2.0`，`loss = w²`，`lr = 0.1`。

**要做什么**：完成一次反向 + `SGD.step()`，把更新后的 `w` 填进 `w_after_9_1`。

**提示**：更新式 `θ ← θ − lr·g`，其中 `g = 2w`。

In [ ]:
w_9_1 = torch.tensor(2.0, requires_grad=True)
opt_9_1 = torch.optim.SGD([w_9_1], lr=0.1)
(w_9_1 ** 2).backward()
opt_9_1.step()
w_after_9_1 = None   # TODO: float(w_9_1.detach())

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("9.1 一步后 w = 1.6", lambda: abs(w_after_9_1 - 1.6) < 1e-6)

### 练习 9.2　Adam 与 AdamW 的权重衰减　★★

**对应手册**：12.2 Adam vs AdamW

两个参数都从 1.0 出发，梯度都显式设成 0，`lr=0.1`、`weight_decay=0.1`：

- `Adam` 把 weight decay 加进梯度后再做自适应缩放。
- `AdamW` 直接对参数做乘性衰减（解耦）。

**要做什么**：分别走一步，把结果填进 `adam_after_9_2` 与 `adamw_after_9_2`。

**提示**：梯度为 0 时，Adam 的更新量约等于 `lr·wd`；AdamW 的参数乘 `(1 − lr·wd)`。

In [ ]:
a_9_2 = torch.tensor(1.0, requires_grad=True)
b_9_2 = torch.tensor(1.0, requires_grad=True)

opt_adam_9_2 = torch.optim.Adam([a_9_2], lr=0.1, weight_decay=0.1)
opt_adamw_9_2 = torch.optim.AdamW([b_9_2], lr=0.1, weight_decay=0.1)

a_9_2.grad = torch.tensor(0.0)
b_9_2.grad = torch.tensor(0.0)
opt_adam_9_2.step()
opt_adamw_9_2.step()

adam_after_9_2 = None    # TODO: float(a_9_2.detach())
adamw_after_9_2 = None   # TODO: float(b_9_2.detach())

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("9.2a Adam 后 w ≈ 0.9", lambda: abs(adam_after_9_2 - 0.9) < 1e-4)
check("9.2b AdamW 后 w = 0.99", lambda: abs(adamw_after_9_2 - 0.99) < 1e-6)

### 练习 9.3　warmup + cosine 调度　★★★

**对应手册**：13.2 正确用法 / 13.3 Warmup

搭建一个 2000 步的训练：

- 前 100 步线性 warmup 到 `1e-3`；
- 后 1900 步余弦退火到 0。

**要做什么**
1. 用 `LambdaLR` + `CosineAnnealingLR` + `SequentialLR(milestones=[100])` 组装调度器。
2. 循环 2000 步：先 `opt.step()`，再 `scheduler.step()`，把 `get_last_lr()[0]` 追加到 `lrs_9_3`。

**提示**：warmup 的 lambda 是 `lambda s: (s + 1) / 100`；`CosineAnnealingLR` 的 `T_max` 是 warmup 之后的步数 1900。

In [ ]:
model_9_3 = nn.Linear(2, 2)
opt_9_3 = torch.optim.AdamW(model_9_3.parameters(), lr=1e-3)

warmup_9_3 = None     # TODO: LambdaLR
cosine_9_3 = None     # TODO: CosineAnnealingLR
scheduler_9_3 = None  # TODO: SequentialLR

lrs_9_3 = []
# TODO: for 2000 步：opt.step() -> scheduler.step() -> 记录 lr

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("9.3a 记录 2000 个 lr", lambda: len(lrs_9_3) == 2000)
check("9.3b warmup 单调升且第 99 步 ≈ 1e-3",
      lambda: all(lrs_9_3[i] <= lrs_9_3[i + 1] + 1e-12 for i in range(99))
      and abs(lrs_9_3[99] - 1e-3) < 1e-8)
check("9.3c 余弦段单调降",
      lambda: all(lrs_9_3[i] >= lrs_9_3[i + 1] - 1e-12 for i in range(100, 1999)))
check("9.3d 末值为 0", lambda: lrs_9_3[-1] < 1e-9)

### 练习 9.4　ReduceLROnPlateau　★★

**对应手册**：13.1 全家福 / 13.2 注意

`SGD(lr=0.1)` + `ReduceLROnPlateau(factor=0.5, patience=2)`，跑 10 个 epoch，每次都喂一个**不变**的指标 `1.0`。

**要做什么**：记录每次 `step(1.0)` 之后的 lr，填进 `lrs_9_4`。

**要问自己**：patience=2 意味着连续几次没改善才降？10 个 epoch 一共降几次？

In [ ]:
torch.manual_seed(10)
m_9_4 = nn.Linear(1, 1)
opt_9_4 = torch.optim.SGD(m_9_4.parameters(), lr=0.1)
sch_9_4 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_9_4, mode="min", factor=0.5, patience=2)

lrs_9_4 = []
# TODO: 10 个 epoch，每次 sch_9_4.step(1.0)，记录 opt_9_4.param_groups[0]["lr"]

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("9.4a 记录 10 个 lr", lambda: len(lrs_9_4) == 10)
check("9.4b 初始 lr 0.1", lambda: abs(lrs_9_4[0] - 0.1) < 1e-12)
check("9.4c 一共衰减 3 次（4 个不同值）",
      lambda: len({round(v, 10) for v in lrs_9_4}) == 4)
check("9.4d 最终 lr = 0.0125", lambda: abs(lrs_9_4[-1] - 0.0125) < 1e-9)

### 练习 9.5　参数组：分层学习率　★★

**对应手册**：12.3 参数组

`TwoHead` 有两个 `Linear(4,4,bias=False)`。给它们建两个参数组：`a` 的 `lr=1e-2`，`b` 的 `lr=1e-3`。

**要做什么**
1. 创建 `opt_9_5`（AdamW，两个参数组）。
2. 给所有参数喂相同梯度（全 1），走一步。
3. 分别算两组权重的平均绝对变化量 `delta_a_9_5`、`delta_b_9_5`，以及比值 `lr_ratio_9_5`。

In [ ]:
class TwoHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Linear(4, 4, bias=False)
        self.b = nn.Linear(4, 4, bias=False)
    def forward(self, x):
        return self.b(self.a(x))

torch.manual_seed(7)
net_9_5 = TwoHead()
opt_9_5 = None   # TODO: AdamW，参数组 [{a: lr=1e-2}, {b: lr=1e-3}]

for p in net_9_5.parameters():
    p.grad = torch.ones_like(p)

before_a_9_5 = net_9_5.a.weight.detach().clone()
before_b_9_5 = net_9_5.b.weight.detach().clone()
# TODO: opt_9_5.step()
delta_a_9_5 = None    # TODO: 平均绝对变化量
delta_b_9_5 = None    # TODO
lr_ratio_9_5 = None   # TODO: delta_a / delta_b

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("9.5a 两组参数都在动", lambda: delta_a_9_5 > 0 and delta_b_9_5 > 0)
check("9.5b 变化量比值 ≈ 10", lambda: abs(lr_ratio_9_5 - 10.0) < 0.1)
check("9.5c 参数组 lr 正确",
      lambda: [g["lr"] for g in opt_9_5.param_groups] == [1e-2, 1e-3])

In [ ]:
scoreboard()

## 第十部分　工程工具（手册第 14–19 章）

从「会写模型」到「能训练、能排错」：梯度裁剪、剪枝、数据管线、保存加载、混合精度。
建议用时 60–90 分钟，全部可在 CPU 上完成。

### 练习 10.1　梯度裁剪　★

**对应手册**：14.1 梯度裁剪

参数梯度为 `[3, 4]`（范数 5）。

**要做什么**
1. 填裁剪前的范数 `norm_before_10_1`。
2. 用 `clip_grad_norm_(..., max_norm=1.0)` 裁剪，把结果填 `grads_after_10_1`。
3. 另一个参数用 `clip_grad_value_(..., 0.5)`，把结果填 `grads_value_10_1`。

In [ ]:
p_10_1 = nn.Parameter(torch.zeros(2))
p_10_1.grad = torch.tensor([3.0, 4.0])
norm_before_10_1 = None   # TODO: float(p_10_1.grad.norm())
# TODO: clip_grad_norm_(p_10_1, max_norm=1.0)
grads_after_10_1 = None   # TODO: p_10_1.grad.clone()

q_10_1 = nn.Parameter(torch.zeros(2))
q_10_1.grad = torch.tensor([3.0, 4.0])
# TODO: clip_grad_value_(q_10_1, 0.5)
grads_value_10_1 = None

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("10.1a 裁剪前范数 5", lambda: abs(norm_before_10_1 - 5.0) < 1e-6)
check("10.1b 范数裁剪为 [0.6,0.8]",
      lambda: close(grads_after_10_1, torch.tensor([0.6, 0.8]))
      and abs(float(grads_after_10_1.norm()) - 1.0) < 1e-6)
check("10.1c 数值裁剪为 [0.5,0.5]",
      lambda: close(grads_value_10_1, torch.tensor([0.5, 0.5])))

### 练习 10.2　剪枝：mask 与固化　★★

**对应手册**：14.4 剪枝

对 `nn.Linear(4,4)` 做 L1 非结构化剪枝，`amount=0.5`。

**要做什么**
1. 剪枝后数一下权重里的 0 有多少个，填 `zeros_after_prune_10_2`。
2. 是否存在 `weight_mask`？填 `has_mask_10_2`。
3. 调用 `prune.remove` 之后再数 0、再看 mask 是否存在，填 `zeros_after_remove_10_2`、`has_mask_after_remove_10_2`。

In [ ]:
torch.manual_seed(11)
layer_10_2 = nn.Linear(4, 4)
prune.l1_unstructured(layer_10_2, name="weight", amount=0.5)
zeros_after_prune_10_2 = None   # TODO
has_mask_10_2 = None            # TODO: bool

prune.remove(layer_10_2, "weight")
zeros_after_remove_10_2 = None      # TODO
has_mask_after_remove_10_2 = None   # TODO: bool

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("10.2a 剪枝后 8 个 0", lambda: zeros_after_prune_10_2 == 8 == int((layer_10_2.weight == 0).sum()))
check("10.2b 存在 weight_mask", lambda: has_mask_10_2 is True)
check("10.2c remove 后 0 的个数不变", lambda: zeros_after_remove_10_2 == 8)
check("10.2d remove 后 mask 被删除", lambda: has_mask_after_remove_10_2 is False)

### 练习 10.3　Dataset / DataLoader / collate　★★

**对应手册**：15.3 DataLoader / 15.4 collate

四条变长序列（长度 3、1、4、2）。批大小 2，`shuffle=False`。

**要做什么**
1. 实现 `collate_10_3`：用 `pad_sequence(batch_first=True)` 补齐序列，用 `torch.tensor` 把标签拼成一批。
2. 迭代 DataLoader，填两个批次的形状 `first_shape_10_3`、`second_shape_10_3`。

**提示**：第一批是长度 3 和 1，第二批是长度 4 和 2。标签是 Python `int`，`torch.stack` 只吃张量，用 `torch.tensor(labels)` 更直接。

In [ ]:
class SeqDataset(Dataset):
    def __init__(self):
        self.data = [
            torch.tensor([1.0, 2.0, 3.0]),
            torch.tensor([4.0]),
            torch.tensor([5.0, 6.0, 7.0, 8.0]),
            torch.tensor([9.0, 10.0]),
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        return self.data[index], index

def collate_10_3(batch):
    seqs, labels = zip(*batch)
    # TODO: 返回 (pad_sequence(seqs, batch_first=True), torch.tensor(labels))
    return None

loader_10_3 = DataLoader(SeqDataset(), batch_size=2, shuffle=False, collate_fn=collate_10_3)
batches_10_3 = list(loader_10_3)
first_shape_10_3 = None    # TODO
second_shape_10_3 = None   # TODO

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("10.3a 第一批 (2,3)",
      lambda: tuple(batches_10_3[0][0].shape) == (2, 3) and first_shape_10_3 == (2, 3))
check("10.3b 第二批 (2,4)",
      lambda: tuple(batches_10_3[1][0].shape) == (2, 4) and second_shape_10_3 == (2, 4))
check("10.3c 补齐位置为 0", lambda: float(batches_10_3[0][0][1, 1:].abs().sum()) == 0.0)
check("10.3d 标签成批", lambda: tuple(batches_10_3[0][1].shape) == (2,))

### 练习 10.4　state_dict 保存与加载　★★

**对应手册**：19.1 两种保存对象 / 19.2 加载

**要做什么**
1. 把 `model_a_10_4.state_dict()` 保存到 `path_10_4`。
2. 新建同结构的 `model_b_10_4`，加载权重并切到 eval，比较两者输出，填 `same_output_10_4`。
3. 取出 state_dict、删掉一个键（`"0.bias"`），用 `strict=False` 再加载一次，把是否成功填进 `strict_false_ok_10_4`。

**提示**：加载用 `torch.load(path, map_location="cpu", weights_only=True)`。

In [ ]:
torch.manual_seed(12)
def make_mlp_10_4():
    return nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 2))

model_a_10_4 = make_mlp_10_4()
x_10_4 = torch.randn(5, 3)
model_a_10_4.eval()

ckpt_dir_10_4 = tempfile.mkdtemp()
path_10_4 = os.path.join(ckpt_dir_10_4, "model.pt")

model_b_10_4 = None
same_output_10_4 = None
strict_false_ok_10_4 = None

# TODO 1: 保存 state_dict
# TODO 2: 新建 model_b_10_4，加载，eval，比较输出
# TODO 3: 删除 "0.bias" 后用 strict=False 加载，填 strict_false_ok_10_4

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("10.4a 加载后输出一致",
      lambda: same_output_10_4 is True and close(model_a_10_4(x_10_4), model_b_10_4(x_10_4)))
check("10.4b strict=False 允许缺键", lambda: strict_false_ok_10_4 is True)
check("10.4c 文件里是普通字典",
      lambda: isinstance(torch.load(path_10_4, map_location="cpu", weights_only=True), dict))

### 练习 10.5　autocast 内外的 dtype　★★

**对应手册**：16.1 两个 API

**要做什么**
1. 在 `torch.autocast(device_type=..., dtype=...)` 内对 `Linear` 前向，结果填 `y_amp_10_5`。
2. 在 autocast 外做同样的前向，填 `y_outside_10_5`。
3. 判断两者的 dtype，填 `amp_ok_10_5`、`outside_ok_10_5`。

**说明**：本机有 CUDA 时用 fp16，没有则退回 CPU + bf16，两种都能练流程。注意**模型和数据都要放到 `amp_device_10_5` 上**：autocast 只影响指定设备上的算子，放在 CPU 上的层仍然是 fp32。

In [ ]:
amp_device_10_5 = "cuda" if torch.cuda.is_available() else "cpu"
amp_dtype_10_5 = torch.float16 if amp_device_10_5 == "cuda" else torch.bfloat16
lin_10_5 = nn.Linear(8, 8).to(amp_device_10_5)
x_10_5 = torch.randn(4, 8, device=amp_device_10_5)

y_amp_10_5 = None        # TODO: 在 torch.autocast(...) 内前向
y_outside_10_5 = None    # TODO
amp_ok_10_5 = None       # TODO: bool
outside_ok_10_5 = None   # TODO: bool

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("10.5a autocast 内输出低精度",
      lambda: y_amp_10_5.dtype in (torch.float16, torch.bfloat16) and amp_ok_10_5 is True)
check("10.5b autocast 外仍是 float32",
      lambda: y_outside_10_5.dtype == torch.float32 and outside_ok_10_5 is True)

### 练习 10.6　AMP 流程与选择题　★★

**对应手册**：16.3 正确顺序 / 16.4 坑

下面这段 AMP 训练步骤**有一处错误**：

```python
optimizer.zero_grad()
with torch.autocast("cuda", dtype=torch.float16):
    loss = criterion(model(x), y)
scaler.scale(loss).backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
scaler.update()
```

**a)** 错在哪？
A. 应该在 clip 之前调用 `scaler.unscale_(optimizer)`
B. 应该用 `optimizer.step(optimizer)`
C. autocast 应该包住 backward
D. 没有问题

**b)** 用 bf16 训练时，GradScaler 应该怎么设置？
A. 必须启用 B. `enabled=False`（bf16 不需要缩放） C. 用 fp64 代替 D. 只能用于 CPU

**c)** scaler 检测到 inf/nan 时会发生什么？
A. 立即停止训练 B. 自动跳过这一步并把 scale 减半 C. 把学习率除以 2 D. 把梯度乘 2

In [ ]:
answer_10_6a = None
answer_10_6b = None
answer_10_6c = None

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
quiz("10.6a AMP 顺序错误", answer_10_6a, "A", "clip 前必须 unscale_，否则裁的是放大后的梯度。")
quiz("10.6b bf16 的 scaler", answer_10_6b, "B", "bf16 指数范围与 fp32 相同，不需要 loss scaling。")
quiz("10.6c 溢出处理", answer_10_6c, "B", "scaler 跳过该步并降低 scale，稳定后再逐步增大。")

In [ ]:
scoreboard()

## 第十一部分　综合实战（手册第 20 章 / 附录 A、B）

四道综合题：完整训练循环、因果 Transformer 块、报错诊断、断点续训。
建议用时 90–120 分钟。

### 练习 11.1　训练一个 MLP 学会 XOR　★★

**对应手册**：12.1 优化器 / 20 章 训练循环

四个点的 XOR 是线性不可分问题，必须用带非线性的 MLP。

**要做什么**：补完训练循环（1500 步）：
`zero_grad → BCEWithLogitsLoss → backward → step`，并把每步 loss 记进 `losses_11_1`。
最后填 `final_loss_11_1`（最后一步 loss）与 `accuracy_11_1`（四点的分类准确率）。

**提示**：模型已是 `Linear(2,8) → Tanh → Linear(8,1)`，优化器 Adam，lr=0.05。

In [ ]:
torch.manual_seed(8)
X_11_1 = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_11_1 = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

model_11_1 = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
opt_11_1 = torch.optim.Adam(model_11_1.parameters(), lr=0.05)

losses_11_1 = []
final_loss_11_1 = None
accuracy_11_1 = None
# TODO: 写 1500 步训练循环

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("11.1a loss 降到 0.01 以下", lambda: final_loss_11_1 < 0.01)
check("11.1b 四个点全部分类正确", lambda: accuracy_11_1 == 1.0)
check("11.1c 记录了 1500 步", lambda: len(losses_11_1) == 1500)

### 练习 11.2　补全因果 Transformer 块　★★★

**对应手册**：10.3 Transformer 模块 / 附录 A.4

块结构已给出：Pre-LN → 多头自注意力 → 残差 → Pre-LN → FFN → 残差。

**要做什么**
1. 填 `self.attn`：`nn.MultiheadAttention(d, heads, batch_first=True)`。
2. 在 forward 里用严格上三角 mask 调用注意力，`need_weights=False`，输出赋给 `a`。
3. 填 `causal_11_2`：修改最后一个 token 后，前 5 个位置的输出是否不变。

**提示**：`nn.MultiheadAttention` 的 bool `attn_mask` 中 **True = 屏蔽**。

In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, d=32, heads=4):
        super().__init__()
        self.norm1 = nn.RMSNorm(d)
        self.attn = None    # TODO: nn.MultiheadAttention(d, heads, batch_first=True)
        self.norm2 = nn.RMSNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.SiLU(), nn.Linear(4 * d, d))
    def forward(self, x):
        L = x.size(1)
        mask = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), diagonal=1)
        h = self.norm1(x)
        a = torch.zeros_like(x)   # TODO: 换成 causal self-attention 的输出
        x = x + a
        return x + self.ffn(self.norm2(x))

block_11_2 = TinyBlock()
block_11_2.eval()
x_11_2 = torch.randn(2, 6, 32)
out_11_2 = block_11_2(x_11_2)

x2_11_2 = x_11_2.clone()
x2_11_2[:, -1] += 5.0
out2_11_2 = block_11_2(x2_11_2)
causal_11_2 = None   # TODO: 前 5 个位置是否不变（bool）

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("11.2a attn 是 MultiheadAttention", lambda: isinstance(block_11_2.attn, nn.MultiheadAttention))
check("11.2b 输出形状 (2,6,32)", lambda: tuple(out_11_2.shape) == (2, 6, 32))
check("11.2c 满足因果性",
      lambda: causal_11_2 is True and close(out_11_2[:, :-1], out2_11_2[:, :-1], atol=1e-5))
check("11.2d 前文确实影响后文",
      lambda: not close(out_11_2[:, 5],
                        block_11_2(torch.cat([x_11_2[:, :1] + 5.0, x_11_2[:, 1:]], dim=1))[:, 5],
                        atol=1e-6))

### 练习 11.3　报错诊断（选择题）　★

**对应手册**：附录 B 常见报错

把答案填进 `answer_11_3a` … `answer_11_3e`。

**a)** `RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x128 and 64x256)` 最可能的原因：
A. 学习率太大　B. 上一层输出维度与本层 `in_features` 不匹配　C. 没有 `zero_grad`　D. 张量是 int

**b)** `CrossEntropyLoss` 抛 `expected scalar type Long but found Float`：
A. 模型输入不是 long　B. target 不是 long　C. 权重不是 float　D. 用了 CUDA

**c)** `Trying to backward through the graph a second time`：
A. 同一张计算图反向两次　B. 没有 zero_grad　C. 梯度爆炸　D. 开了 AMP

**d)** `element 0 of tensors does not require grad and does not have a grad_fn`：
A. 模型没有参数　B. 前向处于 `no_grad` 或输入不需要梯度　C. lr 太小　D. 忘记 `model.eval()`

**e)** `TypeError: can't convert cuda:0 device type tensor to numpy`：
A. 没有先 `.cpu()`　B. 没有优化器　C. numpy 版本太高　D. 张量是 int

In [ ]:
answer_11_3a = None
answer_11_3b = None
answer_11_3c = None
answer_11_3d = None
answer_11_3e = None

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
quiz("11.3a 矩阵形状不匹配", answer_11_3a, "B", "128 与 64 对不上，检查上一层的 out 与本层 in_features。")
quiz("11.3b CE 的 target dtype", answer_11_3b, "B", "分类标签必须是 int64。")
quiz("11.3c 二次反向", answer_11_3c, "A", "每次前向配一次反向；确需多次反向用 retain_graph=True。")
quiz("11.3d 没有 grad_fn", answer_11_3d, "B", "检查是否包在 no_grad / inference_mode 里。")
quiz("11.3e CUDA 张量转 numpy", answer_11_3e, "A", "先 .detach().cpu().numpy()。")

### 练习 11.4　断点续训：完整检查点　★★★

**对应手册**：19.3 checkpoint 应该存什么

训练 5 步后存档，然后模拟两种后续：

- 对照组：原模型、原优化器继续训练 5 步；
- 恢复组：新建模型与优化器，加载检查点后再训练 5 步。

两者权重与优化器状态应当完全一致。

**要做什么**
1. 保存 `{"model": ..., "optim": ..., "step": 5}` 到 `ckpt_path_11_4`。
2. 让 `m_cont_11_4` / `o_cont_11_4` 再训练 5 步。
3. 新建 `m_new_11_4` / `o_new_11_4`，加载检查点，再训练 5 步。
4. 比较权重，填 `resume_matches_11_4`；比较优化器 `exp_avg`，填 `optim_state_restored_11_4`。

In [ ]:
torch.manual_seed(9)
X_11_4 = torch.randn(8, 4)
y_11_4 = torch.randn(8, 1)

def make_model_11_4():
    return nn.Linear(4, 1)

def train_11_4(model, opt, steps):
    for _ in range(steps):
        opt.zero_grad()
        F.mse_loss(model(X_11_4), y_11_4).backward()
        opt.step()

m_cont_11_4 = make_model_11_4()
o_cont_11_4 = torch.optim.AdamW(m_cont_11_4.parameters(), lr=1e-2)
train_11_4(m_cont_11_4, o_cont_11_4, 5)

ckpt_dir_11_4 = tempfile.mkdtemp()
ckpt_path_11_4 = os.path.join(ckpt_dir_11_4, "ckpt.pt")

resume_matches_11_4 = None
optim_state_restored_11_4 = None

# TODO 1: 保存 {"model","optim","step"} 到 ckpt_path_11_4
# TODO 2: 对照组再训练 5 步
# TODO 3: 新建 m_new_11_4 / o_new_11_4，加载检查点，再训练 5 步
# TODO 4: 比较权重与优化器 exp_avg

**自检**（运行这一格；全部 `[PASS]` 即过关，`[FAIL]` 会指出差在哪一项）

In [ ]:
check("11.4a 恢复后权重与不中断训练一致",
      lambda: resume_matches_11_4 is True
      and close(m_new_11_4.weight, m_cont_11_4.weight, atol=1e-6))
check("11.4b 优化器状态一致", lambda: optim_state_restored_11_4 is True)
check("11.4c 检查点包含 model/optim/step",
      lambda: {"model", "optim", "step"}.issubset(
          set(torch.load(ckpt_path_11_4, map_location="cpu", weights_only=True).keys())))

In [ ]:
scoreboard()

In [ ]:
scoreboard()

# 参考答案

> 写法不唯一，判定标准是每题的「自检」格。
> 下面的代码可以直接复制回对应题目的自检格上方运行；答案里的变量名与题目 starter 保持一致。

建议顺序：先自己写 → 跑自检 → 卡住超过 10 分钟 → 再对照答案。

## 第一部分　张量基础（手册第 1–2 章）

### 练习 1.1　dtype 陷阱

Python 整数被推断为 `int64`；`nn.Linear` 的权重是 `float32`，矩阵乘要求两边 dtype 一致，直接传整数会抛 `RuntimeError: mat1 and mat2 must have the same dtype`。模型输入一律 float、索引与标签一律 long，是全书出现频率最高的一条规矩。

In [ ]:
t_int = torch.tensor([[1, 2], [3, 4]])
dtype_1_1 = t_int.dtype
t_float = t_int.float()

try:
    nn.Linear(2, 2)(t_int)
    linear_accepts_int = True
except RuntimeError:
    linear_accepts_int = False

### 练习 1.2　形状预测

`view(2, -1)` 与 `flatten(1)` 形状相同但来源不同：前者要求内存连续，后者必要时会拷贝。`permute(2,0,1)` 把原来的第 2 维提到最前，得到 (4,2,3)；`transpose(1,2)` 只交换第 1、2 维，得到 (2,4,3)。

In [ ]:
x_1_2 = torch.randn(2, 3, 4)

shapes_1_2 = {
    "view": (2, 12),
    "permute": (4, 2, 3),
    "transpose": (2, 4, 3),
    "unsqueeze": (1, 2, 3, 4),
    "flatten": (2, 12),
    "stack": (2, 2, 3, 4),
}

### 练习 1.3　view 报错与修复

`view` 只改元数据，要求元素在内存中连续排列；`permute` 之后 strides 变了但底层存储没动，所以 `view` 报 `RuntimeError: view size is not compatible with input tensor's size and stride`。`contiguous()` 会真正拷贝一份连续内存；`reshape` 则是「能视图就视图，不能就拷贝」。

In [ ]:
x_1_3 = torch.randn(2, 3, 4)
y_1_3 = x_1_3.permute(1, 2, 0)

try:
    y_1_3.view(3, 8)
    err_1_3 = "no error"
except Exception as e:
    err_1_3 = type(e).__name__

z_1_3 = y_1_3.contiguous().view(3, 8)

### 练习 1.4　cat 与 stack

把 `stack` 当 `cat` 用是新手最常见的形状 bug：本来想要 (4,3)，结果拿到 (2,2,3)。`dim=-1` 堆叠相当于在最后一维做「配对」，常用于把 (x, y) 两个特征拼成对。

In [ ]:
a_1_4 = torch.randn(2, 3)
b_1_4 = torch.randn(2, 3)

c_1_4 = torch.cat([a_1_4, b_1_4], dim=0)
s_1_4 = torch.stack([a_1_4, b_1_4], dim=0)
s_last_1_4 = torch.stack([a_1_4, b_1_4], dim=-1)

shapes_1_4 = {"c": (4, 3), "s": (2, 2, 3), "s_last": (2, 3, 2)}

### 练习 1.5　广播规则

`d` 的最后一维 5 与 4 既不相等也没有 1，无法广播，抛 `RuntimeError: The size of tensor a (5) must match the size of tensor b (4)`。注意 `(3,1)+(1,4)` 两个维度都靠 1 扩展，结果 (3,4)——这就是注意力里 mask 与 scores 广播的基础。

In [ ]:
broadcast_1_5 = {
    "a": (3, 4),
    "b": (3, 4),
    "c": (2, 3, 4),
    "d": "error",
}

### 练习 1.6　einsum 写注意力分数

注意力分数的本质就是「每个 query 向量和每个 key 向量做内积」：`D` 维被缩并掉，留下 `(L, S)` 的相关性矩阵。einsum 的好处是维度一写出来就不会错，代价是初学时可读性差；工程代码里两者等价。

In [ ]:
q_1_6 = torch.randn(2, 4, 5, 8)
k_1_6 = torch.randn(2, 4, 6, 8)

scores_einsum = torch.einsum("bhld,bhsd->bhls", q_1_6, k_1_6)
scores_matmul = q_1_6 @ k_1_6.transpose(-1, -2)
eq_1_6 = torch.allclose(scores_einsum, scores_matmul)

### 练习 1.7　因果 mask

softmax 里 `-inf` 的指数是 0，所以被屏蔽位置的概率严格为 0；第 0 个 token 只能看自己，概率必然是 1。这里 mask 的语义是「True = 屏蔽」（`nn.MultiheadAttention` 也是这样），但 `F.scaled_dot_product_attention` 的 bool mask 相反，见练习 7.2。

In [ ]:
L_1_7 = 4
scores_1_7 = torch.randn(L_1_7, L_1_7)

mask_1_7 = torch.triu(torch.ones(L_1_7, L_1_7, dtype=torch.bool), diagonal=1)
masked_1_7 = scores_1_7.masked_fill(mask_1_7, float("-inf"))
probs_1_7 = torch.softmax(masked_1_7, dim=-1)

### 练习 1.8　视图 / 拷贝 / detach

视图和 `detach` 共享同一块内存，但 `detach` 把 `requires_grad` 变成 False，反向时梯度不会流回原张量；`clone` 则相反：新内存、但仍在计算图里。改 `v` 会改 `b`，改 `c` 不会——这就是原地修改导致反向报 `one of the variables needed ... modified by an inplace operation` 的根源。

In [ ]:
b_1_8 = torch.arange(6.).reshape(2, 3).requires_grad_(True)
v_1_8 = b_1_8.view(3, 2)
c_1_8 = b_1_8.clone()
d_1_8 = b_1_8.detach()
r_1_8 = b_1_8.reshape(3, 2)

shares_1_8 = {"view": True, "clone": False, "detach": True, "reshape": True}
graphs_1_8 = {"view": True, "clone": True, "detach": False, "reshape": True}

## 第二部分　自动求导（手册第 3 章）

### 练习 2.1　手算梯度并用 backward 验证

`requires_grad=True` 让 `x` 成为叶子张量；前向时 PyTorch 把每个算子记进计算图，`backward()` 按链式法则倒推，结果写进 `x.grad`。手算「2x+3」得到 1、3、7。

In [ ]:
x_2_1 = torch.tensor([-1.0, 0.0, 2.0], requires_grad=True)
y_2_1 = (x_2_1 ** 2 + 3 * x_2_1).sum()

y_2_1.backward()
grad_2_1 = x_2_1.grad

### 练习 2.2　梯度累加与 zero_grad

`dw²/dw = 2w = 4`，两轮累加成 8；所以每个训练 step 必须 `optimizer.zero_grad()`，否则等效学习率随步数变大。同一张图默认在反向时被释放，第二次会抛 `Trying to backward through the graph a second time`；确实需要多次反向时传 `retain_graph=True`。

In [ ]:
w_2_2 = torch.tensor(2.0, requires_grad=True)
for _ in range(2):
    (w_2_2 ** 2).backward()

grad_after_two = float(w_2_2.grad)

w_2_2.grad = None
(w_2_2 ** 2).backward()
grad_after_reset = float(w_2_2.grad)

x_2_2 = torch.tensor(1.0, requires_grad=True)
y_2_2 = x_2_2 ** 2
y_2_2.backward()
try:
    y_2_2.backward()
    twice_backward_ok = True
except RuntimeError:
    twice_backward_ok = False

### 练习 2.3　no_grad 与 detach

`no_grad` 中的输出 `z` 是叶子张量，`requires_grad=False`，对它反向会抛 `element 0 of tensors does not require grad`。`detach` 只是切断了「这一步」的梯度路径：`loss = 6·w`，所以梯度是 6（而不是 3 倍的 6）。评估与日志计算记得包 `no_grad`，日志里的数字记得 `.detach()`。

In [ ]:
w_2_3 = torch.tensor(2.0, requires_grad=True)
y_2_3 = w_2_3 * 3
with torch.no_grad():
    z_2_3 = w_2_3 * 4

y_req = y_2_3.requires_grad
z_req = z_2_3.requires_grad

z_backward_fails = None
try:
    z_2_3.backward()
    z_backward_fails = False
except RuntimeError:
    z_backward_fails = True

loss_2_3 = (y_2_3.detach() * w_2_3).sum()
loss_2_3.backward()
grad_2_3 = float(w_2_3.grad)

### 练习 2.4　非叶子张量与 retain_grad

`dc/db = 4`、`dc/da = 3×4 = 12`。中间张量的梯度默认算完就丢，是为了省显存；调试梯度流时加 `retain_grad()`。运行这题会看到一条关于「非叶子张量 `.grad`」的 `UserWarning`——这正是本题要说明的现象。这也是 DDP 里「未参与前向的参数」会卡住的原因——没有梯度就没有通信。

In [ ]:
a_2_4 = torch.tensor(2.0, requires_grad=True)
b_2_4 = a_2_4 * 3
c_2_4 = b_2_4 * 4
c_2_4.backward()
b_grad_none = b_2_4.grad is None

a2_2_4 = torch.tensor(2.0, requires_grad=True)
b2_2_4 = a2_2_4 * 3
b2_2_4.retain_grad()
c2_2_4 = b2_2_4 * 4
c2_2_4.backward()
b2_grad = b2_2_4.grad
a2_grad_2_4 = a2_2_4.grad

## 第三部分　Module 骨架与线性层（手册第 4–5 章）

### 练习 3.1　组装 MLP

参数：4×16+16 = 80，16×3+3 = 51，合计 131。`nn.Linear` 对最后一维做仿射变换，前面的维度原样保留，所以 `(2,5,4)` 变成 `(2,5,3)`——这也是 Transformer FFN 对每个 token 独立作用的原因。

In [ ]:
mlp_3_1 = nn.Sequential(
    nn.Linear(4, 16),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(16, 3),
)
param_count_3_1 = sum(p.numel() for p in mlp_3_1.parameters())

### 练习 3.2　参数量手算

`10×20+20 = 220`，`20×5+5 = 105`，合计 325。会手算参数量的意义在于：显存估算、LoRA 的秩选择、判断该用 DDP 还是 FSDP，都从「模型多大」开始。

In [ ]:
model_3_2 = nn.Sequential(
    nn.Linear(10, 20), nn.ReLU(),
    nn.Linear(20, 5),
)

params_3_2 = 325
computed_3_2 = sum(p.numel() for p in model_3_2.parameters())

### 练习 3.3　register_buffer

buffer 的四个特征：进 `state_dict`、随 `.to()` 搬设备、不参与梯度、不进优化器。BN 的 `running_mean`、RoPE 的 cos/sin 表都是 buffer。普通属性则像「贴在模型上的便签」，序列化和搬家时都会丢。

In [ ]:
class Scaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("scale", torch.tensor(2.0))
    def forward(self, x):
        return x * self.scale

scaler_3_3 = Scaler()

### 练习 3.4　ModuleList 与普通 list

普通 list 里的模块不会被 `nn.Module.__setattr__` 识别为子模块：它们不出现在 `parameters()` / `state_dict()` 里，`.to(device)` 也不会搬它们，优化器更看不到——模型「能跑但不训练」的经典 bug。`ModuleDict` 同理，用于多分支。

In [ ]:
class Stack(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([nn.Linear(4, 4) for _ in range(3)])
    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return x

stack_3_4 = Stack()
modulelist_param_count = len(list(stack_3_4.parameters()))

### 练习 3.5　冻结底座并只优化头部

冻结后 backbone 不会产生 `.grad`，即使误传进优化器也不会更新（优化器会跳过 `grad is None` 的参数）。迁移学习里冻结底座 + 训练新头是标准做法；要更细的控制就用参数组（练习 9.5）或 LoRA。

In [ ]:
class Net35(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Linear(8, 8)
        self.head = nn.Linear(8, 2)
    def forward(self, x):
        return self.head(self.backbone(x))

net_3_5 = Net35()
for p in net_3_5.backbone.parameters():
    p.requires_grad_(False)
opt_3_5 = torch.optim.AdamW(net_3_5.head.parameters(), lr=1e-3)

### 练习 3.6　LazyLinear 的初始化时机

Lazy 模块第一次前向才创建真实参数，适合不确定输入维度的原型。但在第一次 forward 之前不要取 `weight` 的数值（会报未初始化错误），保存 state_dict 也要注意时机；正式代码建议显式写下维度，可读性更好。

In [ ]:
from torch.nn.parameter import UninitializedParameter

lazy_3_6 = nn.LazyLinear(3)
before_forward_uninitialized = any(
    isinstance(p, UninitializedParameter) for p in lazy_3_6.parameters())

_ = lazy_3_6(torch.randn(5, 4))
after_weight_shape = tuple(lazy_3_6.weight.shape)

## 第四部分　卷积与池化（手册第 6 章）

### 练习 4.1　卷积输出尺寸

a 是 same padding：`(32+2-2-1)+1 = 32`；b 是 31/2 向下取整 15；c 是 (32+2-2-1)/2+1 = 16；d 的有效核是 `2×(3-1)+1 = 5`，`32-5+1 = 28`。CNN 里绝大多数 shape 报错都源于这一个公式算错。

In [ ]:
out_heights_4_1 = {
    "a": 32,
    "b": 15,
    "c": 16,
    "d": 28,
}

### 练习 4.2　卷积参数量与 groups

a：8×3×9+8 = 224；b：每组只连 1 个输入通道、输出 2 个通道，6×1×9+6 = 60；c：3×1×9+3 = 30。`groups=C_in=C_out` 就是 depthwise 卷积，配合 1×1 的 pointwise 卷积即深度可分离卷积，参数量与计算量都降一个量级。

In [ ]:
conv_params_4_2 = {"a": 224, "b": 60, "c": 30}

### 练习 4.3　CNN 分类头与全局平均池化

参数：224 + 1168 + 170 = 1562。`AdaptiveAvgPool2d(1)` 把任意分辨率压成 `(N,C,1,1)`，所以分类头只与通道数有关；换成 224×224 输入也不用改 head，这是分类网络的标准收尾。

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(16, num_classes)
    def forward(self, x):
        return self.head(self.features(x).flatten(1))

cnn_4_3 = SmallCNN()
out_4_3 = cnn_4_3(torch.randn(2, 3, 32, 32))
pooled_shape_4_3 = tuple(cnn_4_3.features(torch.randn(2, 3, 32, 32)).shape)
param_count_4_3 = sum(p.numel() for p in cnn_4_3.parameters())

### 练习 4.4　用梯度实测感受野

感受野 1 → 3 → 5 → 7。梯度非零的位置恰好是「影响这个输出的输入区域」，用 backward 实测比死记公式可靠。两层 3×3 的感受野等于一层 5×5，但参数更少、多一次非线性，所以 VGG 之后大家都堆小卷积。

In [ ]:
net_4_4 = nn.Sequential(
    nn.Conv2d(1, 1, 3), nn.Conv2d(1, 1, 3), nn.Conv2d(1, 1, 3),
)
x_4_4 = torch.randn(1, 1, 9, 9, requires_grad=True)
y_4_4 = net_4_4(x_4_4)
y_4_4[0, 0, 1, 1].backward()

rf_4_4 = 7
measured_4_4 = int((x_4_4.grad[0, 0].abs() > 0).sum())

## 第五部分　循环网络（手册第 7 章）

### 练习 5.1　batch_first 与 LSTM 输出形状

PyTorch 循环层默认序列在前，与 DataLoader 习惯相反，是全库最反直觉的默认值。`h_n` 的形状永远是 `(层数×方向数, N, hidden)`，与 batch_first 无关。MiniMind 里必须显式写 `batch_first=True`。

In [ ]:
lstm_5_1 = nn.LSTM(16, 32)
out_5_1, (h_5_1, c_5_1) = lstm_5_1(torch.randn(10, 2, 16))

lstm_bf_5_1 = nn.LSTM(16, 32, batch_first=True)
out_bf_5_1, (h_bf_5_1, c_bf_5_1) = lstm_bf_5_1(torch.randn(2, 10, 16))

shapes_5_1 = {"out": (10, 2, 32), "h": (1, 2, 32),
              "out_bf": (2, 10, 32), "h_bf": (1, 2, 32)}
lstm_default_batch_first = False

### 练习 5.2　pad_sequence 与 pack_padded_sequence

pack 后只对真实 token 计算：3+1+2 = 6 个时间步，`data` 形状 `(6, hidden)`。不 pack 的话 padding 会参与循环、污染最后一步的隐状态。`enforce_sorted=False` 允许长度乱序（内部会排序并记住逆序）。

In [ ]:
seqs_5_2 = [torch.tensor([1.0, 2.0, 3.0]), torch.tensor([4.0]), torch.tensor([5.0, 6.0])]

padded_5_2 = nn.utils.rnn.pad_sequence(seqs_5_2, batch_first=True)
lengths_5_2 = torch.tensor([3, 1, 2])
packed_5_2 = nn.utils.rnn.pack_padded_sequence(
    padded_5_2.unsqueeze(-1), lengths_5_2, batch_first=True, enforce_sorted=False)

gru_5_2 = nn.GRU(1, 8, batch_first=True)
packed_out_5_2, _ = gru_5_2(packed_5_2)
packed_data_shape_5_2 = tuple(packed_out_5_2.data.shape)

### 练习 5.3　GRU 文本分类器

参数：Embedding 800 + GRU 4800（3 组 weight_ih/weight_hh 与两组 bias）+ head 66 = 5666。取 `out[:, -1]` 只在定长、无 padding 时正确；变长序列应使用 `pack_padded_sequence` 后的 `h_n`。

In [ ]:
class TextClf(nn.Module):
    def __init__(self, vocab=50, d=16, hidden=32, classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=0)
        self.rnn = nn.GRU(d, hidden, batch_first=True)
        self.head = nn.Linear(hidden, classes)
    def forward(self, ids):
        out, _ = self.rnn(self.emb(ids))
        return self.head(out[:, -1])

clf_5_3 = TextClf()
logits_5_3 = clf_5_3(torch.tensor([[1, 2, 3], [4, 5, 6]]))
logits_shape_5_3 = tuple(logits_5_3.shape)
param_count_5_3 = sum(p.numel() for p in clf_5_3.parameters())

### 练习 5.4　手写 LSTM 单步

门的顺序 i/f/g/o 是 PyTorch 的约定（不一定与论文或别的框架相同）。LSTM 的关键是细胞状态的**加法**更新：`c' = f⊙c + i⊙g`，梯度沿这条「高速公路」回传，f 接近 1 时不会指数衰减。

In [ ]:
torch.manual_seed(1)
cell_5_4 = nn.LSTMCell(8, 16)
x_5_4 = torch.randn(2, 8)
h0_5_4 = torch.randn(2, 16)
c0_5_4 = torch.randn(2, 16)
h_ref_5_4, c_ref_5_4 = cell_5_4(x_5_4, (h0_5_4, c0_5_4))

gates_5_4 = (x_5_4 @ cell_5_4.weight_ih.T + cell_5_4.bias_ih
             + h0_5_4 @ cell_5_4.weight_hh.T + cell_5_4.bias_hh)
i_pre, f_pre, g_pre, o_pre = gates_5_4.chunk(4, dim=1)
i_5_4 = torch.sigmoid(i_pre)
f_5_4 = torch.sigmoid(f_pre)
g_5_4 = torch.tanh(g_pre)
o_5_4 = torch.sigmoid(o_pre)
c_5_4 = f_5_4 * c0_5_4 + i_5_4 * g_5_4
h_5_4 = o_5_4 * torch.tanh(c_5_4)

## 第六部分　归一化、激活与 Dropout（手册第 8–9 章）

### 练习 6.1　手写 LayerNorm

`unbiased=False` 对应总体方差（除以 n 而不是 n-1），与 PyTorch 的归一化实现一致。γ、β 让网络能「撤销」归一化，保留表达能力；没有它们，归一化会硬性限制特征尺度。

In [ ]:
torch.manual_seed(13)
x_6_1 = torch.randn(4, 6)
ln_6_1 = nn.LayerNorm(6)
y_ref_6_1 = ln_6_1(x_6_1)

mu = x_6_1.mean(dim=-1, keepdim=True)
var = x_6_1.var(dim=-1, unbiased=False, keepdim=True)
y_manual_6_1 = (x_6_1 - mu) / torch.sqrt(var + 1e-5) * ln_6_1.weight + ln_6_1.bias

### 练习 6.2　手写 RMSNorm

RMSNorm 去掉了「减均值」和「β」，计算更省，效果在 LLaMA/Qwen 里被验证与大模型足够匹配。LLaMA 系列、MiniMind 都默认用它。注意 `eps=None` 时会用当前 dtype 的极小值，fp16 下容易出问题，显式给 eps 更稳。

In [ ]:
torch.manual_seed(14)
x_6_2 = torch.randn(4, 6)
rms_6_2 = nn.RMSNorm(6, eps=1e-6)
y_ref_6_2 = rms_6_2(x_6_2)

y_manual_6_2 = x_6_2 / torch.sqrt(x_6_2.pow(2).mean(dim=-1, keepdim=True) + 1e-6) * rms_6_2.weight

### 练习 6.3　BatchNorm 的训练与推理

训练用当前 batch 统计并更新滑动平均：`running = 0.9 × running + 0.1 × batch_stat`；推理用滑动平均，所以 batch 大小变化不影响输出。忘记 `model.eval()` 是推理结果抖动、指标对不上的头号原因。batch=1 训练时方差为 0，直接报 `Expected more than 1 value per channel`。

In [ ]:
torch.manual_seed(2)
bn_6_3 = nn.BatchNorm1d(3)
before_rm_6_3 = bn_6_3.running_mean.clone()
bn_6_3.train()
_ = bn_6_3(torch.randn(8, 3) * 2 + 5)

running_changed_6_3 = not torch.equal(bn_6_3.running_mean, before_rm_6_3)

bn_6_3.eval()
x_eval_6_3 = torch.randn(8, 3) * 10 + 100
out_eval_6_3 = bn_6_3(x_eval_6_3)
expected_eval_6_3 = ((x_eval_6_3 - bn_6_3.running_mean)
                     / torch.sqrt(bn_6_3.running_var + bn_6_3.eps)
                     * bn_6_3.weight + bn_6_3.bias)
eval_uses_running_6_3 = torch.allclose(out_eval_6_3, expected_eval_6_3, atol=1e-6)

bn_6_3.train()
try:
    bn_6_3(torch.randn(1, 3))
    batch1_raises = False
except ValueError:
    batch1_raises = True

### 练习 6.4　Dropout 的期望保持

inverted dropout：训练时把保留的值除以 `1-p`，所以均值不变、推理时直接直通，不需要额外缩放。这也是为什么 `F.dropout(x)` 默认 `training=True`——评估时忘了传 `training=False` 会悄悄继续丢神经元。

In [ ]:
torch.manual_seed(3)
drop_6_4 = nn.Dropout(0.5)
drop_6_4.train()
x_6_4 = torch.ones(100_000)
y_6_4 = drop_6_4(x_6_4)

drop_scale_6_4 = float(y_6_4.max())
zero_frac_6_4 = float((y_6_4 == 0).float().mean())
mean_6_4 = bool(abs(float(y_6_4.mean()) - 1.0) < 0.01)

drop_6_4.eval()
y_eval_6_4 = drop_6_4(x_6_4)
eval_identity_6_4 = bool(torch.equal(y_eval_6_4, x_6_4))

### 练习 6.5　手写 GELU

GELU 在 0 附近平滑、负半轴不完全归零（保留一点负值），比 ReLU 更好优化；BERT/GPT-2 用精确版或 tanh 近似（`approximate='tanh'`）。大模型 FFN 里另一选择是 SiLU（`x·σ(x)`），SwiGLU 就是它的门控版。

In [ ]:
x_6_5 = torch.linspace(-4, 4, 17)
gelu_ref_6_5 = F.gelu(x_6_5)

gelu_manual_6_5 = x_6_5 * 0.5 * (1 + torch.erf(x_6_5 / math.sqrt(2)))

### 练习 6.6　激活与模式选择（选择题）

softmax 是归一化不是激活；GELU 是现代 Transformer 的默认；ReLU 死亡靠 LeakyReLU / GELU 或调小学习率缓解；推理抖动几乎总是忘了 `eval()`（Dropout 仍在随机丢、BN 用 batch 统计）。

In [ ]:
answer_6_6a = "B"
answer_6_6b = "C"
answer_6_6c = "C"
answer_6_6d = "B"

## 第七部分　Embedding 与注意力（手册第 10 章）

### 练习 7.1　Embedding 就是查表

数学上 Embedding 就是 one-hot 乘权重矩阵，实现上直接按索引取值，省掉稀疏 one-hot 的存储与计算。参数量是 `num_embeddings × dim`；`padding_idx` 的行初始化为 0 且梯度为 0。语言模型常把 `embed_tokens.weight` 与 `lm_head.weight` 共享（MiniMind 的 `tie_word_embeddings=True`）。

In [ ]:
emb_7_1 = nn.Embedding(10, 4, padding_idx=0)
ids_7_1 = torch.tensor([[1, 2, 0], [3, 0, 0]])
out_7_1 = emb_7_1(ids_7_1)

emb_out_shape_7_1 = tuple(out_7_1.shape)
pad_zero_7_1 = bool(float(emb_7_1.weight[0].abs().sum()) == 0.0)
onehot_out_7_1 = F.one_hot(ids_7_1, num_classes=10).float() @ emb_7_1.weight

### 练习 7.2　手写缩放点积注意力

除以 sqrt(d_k) 把点积方差拉回 1 附近，避免 softmax 饱和、梯度消失。`F.scaled_dot_product_attention` 是融合实现（自动选 Flash / Mem-efficient 后端），快在减少显存读写。Mask 语义是三处不一致的重灾区：`nn.MultiheadAttention` 的 bool mask True = 屏蔽，SDPA 的 bool mask True = 参与；浮点 mask 则直接加到分数上。

In [ ]:
def sdpa_manual(q, k, v, attn_mask=None):
    d = q.size(-1)
    scores = q @ k.transpose(-1, -2) / math.sqrt(d)
    if attn_mask is not None:
        scores = scores.masked_fill(attn_mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    return attn @ v

q_7_2 = torch.randn(2, 2, 5, 8)
k_7_2 = torch.randn(2, 2, 5, 8)
v_7_2 = torch.randn(2, 2, 5, 8)
mask_7_2 = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)

out_plain_7_2 = sdpa_manual(q_7_2, k_7_2, v_7_2)
out_causal_7_2 = sdpa_manual(q_7_2, k_7_2, v_7_2, mask_7_2)

### 练习 7.3　多头拆分与合并

多头不是复制数据，而是把 E 维切成 H 份：`reshape(B,L,H,d).permute(0,2,1,3)`。每个头在同一组 token 上用不同的 Q/K/V 投影做注意力，最后拼接再做一次线性变换。`permute` 后内存不连续，最后的 `reshape` 会自动拷贝，这是需要注意的性能点。

In [ ]:
x_7_3 = torch.randn(2, 5, 8)
B_7_3, L_7_3, E_7_3 = x_7_3.shape
H_7_3 = 2

x_split_7_3 = x_7_3.reshape(B_7_3, L_7_3, H_7_3, E_7_3 // H_7_3)
x_heads_7_3 = x_split_7_3.permute(0, 2, 1, 3)
x_merged_7_3 = x_heads_7_3.permute(0, 2, 1, 3).reshape(B_7_3, L_7_3, E_7_3)
shapes_7_3 = {"split": (2, 5, 2, 4), "heads": (2, 2, 5, 4), "merged": (2, 5, 8)}

### 练习 7.4　MultiheadAttention 的形状

和 LSTM 一样，`MultiheadAttention` 默认序列维在前；`batch_first=True` 只是帮你在内部转置。`weights` 形状 `(N, L, S)`，不带 `heads` 维（`need_weights=False` 可省去计算与内存）。`embed_dim % num_heads != 0` 会直接报错。

In [ ]:
mha_7_4 = nn.MultiheadAttention(8, 2)
q_7_4 = torch.randn(5, 3, 8)
kv_7_4 = torch.randn(6, 3, 8)
out_7_4, w_7_4 = mha_7_4(q_7_4, kv_7_4, kv_7_4)
shapes_7_4 = {"out": (5, 3, 8), "weights": (3, 5, 6)}

mha_bf_7_4 = nn.MultiheadAttention(8, 2, batch_first=True)
out_bf_7_4, w_bf_7_4 = mha_bf_7_4(
    torch.randn(3, 5, 8), torch.randn(3, 6, 8), torch.randn(3, 6, 8))
shapes_bf_7_4 = {"out": (3, 5, 8), "weights": (3, 5, 6)}

### 练习 7.5　简化版 RoPE

RoPE 用旋转矩阵编码绝对位置，但旋转后的内积只依赖相对位置（`q_m·k_n = f(m−n)`），所以能外推到训练时没见过的长度。`cos/sin` 预计算一次、存成 buffer，前向只做逐元素乘加——MiniMind 的 `precompute_freqs_cis` 与 `apply_rotary_pos_emb` 就是这个思路。

In [ ]:
def precompute_freqs(dim, max_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_len).float()
    angles = torch.outer(t, freqs)
    return torch.cos(angles), torch.sin(angles)

def apply_rope(x, cos, sin):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

cos_7_5, sin_7_5 = precompute_freqs(8, 16)
x_rope_7_5 = torch.randn(16, 8)
q_rope_7_5 = apply_rope(x_rope_7_5, cos_7_5, sin_7_5)

a_7_5 = torch.randn(8)
b_7_5 = torch.randn(8)
qa_7_5 = apply_rope(a_7_5.expand(16, 8), cos_7_5, sin_7_5)
kb_7_5 = apply_rope(b_7_5.expand(16, 8), cos_7_5, sin_7_5)
D_7_5 = qa_7_5 @ kb_7_5.T
relative_ok_7_5 = bool(torch.allclose(D_7_5[:-1, :-1], D_7_5[1:, 1:], atol=1e-4, rtol=1e-4))

## 第八部分　损失函数（手册第 11 章）

### 练习 8.1　交叉熵 = NLL(log_softmax)

`CrossEntropyLoss` 内部就是 `log_softmax + NLLLoss`，所以它接收的是 **logits**。如果在它前面手动加 softmax，等于对概率再做一次 softmax，梯度被压缩、收敛变慢——这是最常见的损失误用。

In [ ]:
torch.manual_seed(5)
logits_8_1 = torch.randn(4, 3)
target_8_1 = torch.tensor([0, 2, 1, 1])

loss_ce_8_1 = nn.CrossEntropyLoss()(logits_8_1, target_8_1)
loss_nll_8_1 = nn.NLLLoss()(F.log_softmax(logits_8_1, dim=-1), target_8_1)
loss_manual_8_1 = (-F.log_softmax(logits_8_1, dim=-1)[torch.arange(4), target_8_1]).mean()

### 练习 8.2　标签平滑手算

标签平滑限制模型输出过度自信，提升泛化；大模型预训练也常用（例如 epsilon=0.1）。实现上要除以类别数 C，因为平滑后的目标分布总和仍为 1。

In [ ]:
torch.manual_seed(5)
logits_8_2 = torch.randn(4, 3)
target_8_2 = torch.tensor([0, 2, 1, 1])

eps_8_2 = 0.1
loss_ls_8_2 = nn.CrossEntropyLoss(label_smoothing=eps_8_2)(logits_8_2, target_8_2)
logp_8_2 = F.log_softmax(logits_8_2, dim=-1)
C_8_2 = logits_8_2.size(-1)
loss_manual_8_2 = -(
    (1 - eps_8_2) * logp_8_2[torch.arange(4), target_8_2]
    + eps_8_2 / C_8_2 * logp_8_2.sum(dim=-1)
).mean()

### 练习 8.3　ignore_index 与全忽略的 nan

语言模型把 padding 位置标成 -100、用 `ignore_index` 跳过；如果某个 batch 全是 padding，分母为 0 就得到 nan。训练里看到 nan，先查 mask 与 ignore_index，再查学习率与除零。

In [ ]:
logits_8_3 = torch.randn(3, 3)
target_8_3 = torch.tensor([0, 1, -100])

loss_ii_8_3 = nn.CrossEntropyLoss(ignore_index=-100)(logits_8_3, target_8_3)
loss_manual_8_3 = F.cross_entropy(logits_8_3[:2], target_8_3[:2])
loss_all_ignored_8_3 = nn.CrossEntropyLoss(ignore_index=-100)(
    logits_8_3, torch.full_like(target_8_3, -100))
all_ignored_is_nan_8_3 = bool(torch.isnan(loss_all_ignored_8_3))

### 练习 8.4　BCEWithLogits 与 BCELoss

`BCEWithLogitsLoss` 把 sigmoid 与 log 用 log-sum-exp 技巧融合，数值稳定，是二分类/多标签的默认选择。用 `BCELoss` 却直接传 logits 会得到错误结果，且 log 可能作用在负数上。

In [ ]:
torch.manual_seed(6)
logits_8_4 = torch.randn(6)
target_8_4 = torch.tensor([0.0, 1.0, 0.0, 1.0, 1.0, 0.0])

loss_bce_logits_8_4 = nn.BCEWithLogitsLoss()(logits_8_4, target_8_4)
loss_bce_8_4 = nn.BCELoss()(torch.sigmoid(logits_8_4), target_8_4)

### 练习 8.5　KLDivLoss 的 reduction

参数顺序是 `KLDivLoss(input=log 概率, target=概率)`；把 log 概率当 target 会得到 nan（内部要算 `log(target)`）。做分布对齐（蒸馏、RLHF 的 KL 惩罚）时要用 `batchmean`，否则损失会随类别数缩放；2.6 里 `mean` 还会打印一条「未来将改成与 batchmean 一致」的警告，正好说明了这个坑。KL 不对称：`KL(p‖q) ≠ KL(q‖p)`。

In [ ]:
torch.manual_seed(15)
p_logits_8_5 = torch.randn(4, 5)
q_logits_8_5 = torch.randn(4, 5)
logp_8_5 = F.log_softmax(p_logits_8_5, dim=-1)
logq_8_5 = F.log_softmax(q_logits_8_5, dim=-1)

p_prob_8_5 = logp_8_5.exp()
kl_batchmean_8_5 = nn.KLDivLoss(reduction="batchmean")(logq_8_5, p_prob_8_5)
kl_mean_8_5 = nn.KLDivLoss(reduction="mean")(logq_8_5, p_prob_8_5)
kl_manual_8_5 = (p_prob_8_5 * (logp_8_5 - logq_8_5)).sum(dim=-1).mean()

## 第九部分　优化器与学习率（手册第 12–13 章）

### 练习 9.1　SGD 走一步

`2 − 0.1×4 = 1.6`。动量版维护速度 `v = μv + g`，能平滑振荡、越过狭长谷底；Nesterov 先按动量走一步再取梯度，收敛更快。优化器只读取 `.grad`，不关心 loss 怎么来的。

In [ ]:
w_9_1 = torch.tensor(2.0, requires_grad=True)
opt_9_1 = torch.optim.SGD([w_9_1], lr=0.1)
(w_9_1 ** 2).backward()
opt_9_1.step()
w_after_9_1 = float(w_9_1.detach())

### 练习 9.2　Adam 与 AdamW 的权重衰减

Adam 的 L2 正则混在梯度里，被二阶矩归一化后实际衰减量受自适应学习率影响；AdamW 的衰减与 lr 解耦，`w ← w − lr·wd·w`。大模型默认 AdamW，就是因为正则强度可预测、不随梯度尺度漂移。实践里 bias 与 LayerNorm 参数通常不衰减。

In [ ]:
a_9_2 = torch.tensor(1.0, requires_grad=True)
b_9_2 = torch.tensor(1.0, requires_grad=True)

opt_adam_9_2 = torch.optim.Adam([a_9_2], lr=0.1, weight_decay=0.1)
opt_adamw_9_2 = torch.optim.AdamW([b_9_2], lr=0.1, weight_decay=0.1)

a_9_2.grad = torch.tensor(0.0)
b_9_2.grad = torch.tensor(0.0)
opt_adam_9_2.step()
opt_adamw_9_2.step()

adam_after_9_2 = float(a_9_2.detach())
adamw_after_9_2 = float(b_9_2.detach())

### 练习 9.3　warmup + cosine 调度

warmup 让训练初期用很小的 lr：Adam 的二阶矩、BN 的 running 统计在前几步都还不可靠，直接上大 lr 容易发散。顺序必须是 `optimizer.step()` 之后 `scheduler.step()`，否则学习率整体错位一步。torch 2.6 下 `SequentialLR` 可能打印 epoch deprecation warning，可忽略。

In [ ]:
model_9_3 = nn.Linear(2, 2)
opt_9_3 = torch.optim.AdamW(model_9_3.parameters(), lr=1e-3)

warmup_9_3 = torch.optim.lr_scheduler.LambdaLR(opt_9_3, lambda s: (s + 1) / 100)
cosine_9_3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_9_3, T_max=1900, eta_min=0.0)
scheduler_9_3 = torch.optim.lr_scheduler.SequentialLR(
    opt_9_3, [warmup_9_3, cosine_9_3], milestones=[100])

lrs_9_3 = []
for _ in range(2000):
    opt_9_3.step()
    scheduler_9_3.step()
    lrs_9_3.append(scheduler_9_3.get_last_lr()[0])

### 练习 9.4　ReduceLROnPlateau

patience=2 表示容忍 2 次不改善，第 3 次仍不改善才降 lr，所以 10 次调用在第 4、7、10 次各降一半，最终 `0.1 × 0.5³ = 0.0125`。这个调度器**必须传验证指标**，且通常放在 epoch 级而不是 step 级。

In [ ]:
torch.manual_seed(10)
m_9_4 = nn.Linear(1, 1)
opt_9_4 = torch.optim.SGD(m_9_4.parameters(), lr=0.1)
sch_9_4 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_9_4, mode="min", factor=0.5, patience=2)

lrs_9_4 = []
for _ in range(10):
    sch_9_4.step(1.0)
    lrs_9_4.append(opt_9_4.param_groups[0]["lr"])

### 练习 9.5　参数组：分层学习率

梯度恒定时 Adam 单步更新量约等于 lr，所以比值就是学习率之比 10。参数组是迁移学习「底座小 lr、新头大 lr」的原生支持；组内没写的超参（betas、weight_decay）继承外层默认值。

In [ ]:
class TwoHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Linear(4, 4, bias=False)
        self.b = nn.Linear(4, 4, bias=False)
    def forward(self, x):
        return self.b(self.a(x))

torch.manual_seed(7)
net_9_5 = TwoHead()
opt_9_5 = torch.optim.AdamW([
    {"params": net_9_5.a.parameters(), "lr": 1e-2},
    {"params": net_9_5.b.parameters(), "lr": 1e-3},
])

for p in net_9_5.parameters():
    p.grad = torch.ones_like(p)

before_a_9_5 = net_9_5.a.weight.detach().clone()
before_b_9_5 = net_9_5.b.weight.detach().clone()
opt_9_5.step()
delta_a_9_5 = (before_a_9_5 - net_9_5.a.weight.detach()).abs().mean().item()
delta_b_9_5 = (before_b_9_5 - net_9_5.b.weight.detach()).abs().mean().item()
lr_ratio_9_5 = delta_a_9_5 / delta_b_9_5

## 第十部分　工程工具（手册第 14–19 章）

### 练习 10.1　梯度裁剪

`clip_grad_norm_` 按总范数等比缩放，保持方向；`clip_grad_value_` 逐元素截断，会改变方向。混合精度训练里必须先 `scaler.unscale_(optimizer)` 再裁剪，否则裁剪的对象是放大后的梯度。

In [ ]:
p_10_1 = nn.Parameter(torch.zeros(2))
p_10_1.grad = torch.tensor([3.0, 4.0])
norm_before_10_1 = float(p_10_1.grad.norm())
torch.nn.utils.clip_grad_norm_(p_10_1, max_norm=1.0)
grads_after_10_1 = p_10_1.grad.clone()

q_10_1 = nn.Parameter(torch.zeros(2))
q_10_1.grad = torch.tensor([3.0, 4.0])
torch.nn.utils.clip_grad_value_(q_10_1, 0.5)
grads_value_10_1 = q_10_1.grad.clone()

### 练习 10.2　剪枝：mask 与固化

剪枝不改原权重，而是保存 `weight_orig` + 0/1 mask，前向时相乘；`remove` 把 mask 固化回 `weight` 并删掉 mask 属性，便于导出。非结构化剪枝只产生稀疏权重，不会自动加速；结构化剪枝（整个通道/头）才能真正提速。

In [ ]:
torch.manual_seed(11)
layer_10_2 = nn.Linear(4, 4)
prune.l1_unstructured(layer_10_2, name="weight", amount=0.5)
zeros_after_prune_10_2 = int((layer_10_2.weight == 0).sum())
has_mask_10_2 = hasattr(layer_10_2, "weight_mask")

prune.remove(layer_10_2, "weight")
zeros_after_remove_10_2 = int((layer_10_2.weight == 0).sum())
has_mask_after_remove_10_2 = hasattr(layer_10_2, "weight_mask")

### 练习 10.3　Dataset / DataLoader / collate

Dataset 管单条、Sampler 管顺序、DataLoader 管成批。默认的 `default_collate` 要求同字段形状一致，变长序列必须自定义 collate（通常配 `pad_sequence` + `pack_padded_sequence`）。Windows 上 `num_workers>0` 需要 `if __name__ == "__main__"` 保护，最佳起点是 `num_workers=0`。

In [ ]:
class SeqDataset(Dataset):
    def __init__(self):
        self.data = [
            torch.tensor([1.0, 2.0, 3.0]),
            torch.tensor([4.0]),
            torch.tensor([5.0, 6.0, 7.0, 8.0]),
            torch.tensor([9.0, 10.0]),
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, index):
        return self.data[index], index

def collate_10_3(batch):
    seqs, labels = zip(*batch)
    return nn.utils.rnn.pad_sequence(seqs, batch_first=True), torch.tensor(labels)

loader_10_3 = DataLoader(SeqDataset(), batch_size=2, shuffle=False, collate_fn=collate_10_3)
batches_10_3 = list(loader_10_3)
first_shape_10_3 = tuple(batches_10_3[0][0].shape)
second_shape_10_3 = tuple(batches_10_3[1][0].shape)

### 练习 10.4　state_dict 保存与加载

生产环境存 `state_dict` 而不是整个模型：结构由代码定义、权重只是键值对，跨版本更安全。`map_location` 解决「GPU 存的权重在 CPU 机器上加载」，`weights_only=True`（2.6 默认）避免 pickle 执行任意代码。`strict=False` 只放宽键匹配，形状不对仍会报错。

In [ ]:
torch.manual_seed(12)
def make_mlp_10_4():
    return nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 2))

model_a_10_4 = make_mlp_10_4()
x_10_4 = torch.randn(5, 3)
model_a_10_4.eval()

ckpt_dir_10_4 = tempfile.mkdtemp()
path_10_4 = os.path.join(ckpt_dir_10_4, "model.pt")

torch.save(model_a_10_4.state_dict(), path_10_4)

model_b_10_4 = make_mlp_10_4()
model_b_10_4.load_state_dict(torch.load(path_10_4, map_location="cpu", weights_only=True))
model_b_10_4.eval()
same_output_10_4 = bool(torch.allclose(model_a_10_4(x_10_4), model_b_10_4(x_10_4)))

sd_10_4 = torch.load(path_10_4, map_location="cpu", weights_only=True)
sd_10_4.pop("0.bias")
try:
    model_b_10_4.load_state_dict(sd_10_4, strict=False)
    strict_false_ok_10_4 = True
except RuntimeError:
    strict_false_ok_10_4 = False

### 练习 10.5　autocast 内外的 dtype

autocast 只把部分算子（矩阵乘、卷积）降到低精度，归一化、softmax、损失等仍用 fp32，所以叫「混合」精度。fp16 的指数位只有 5 位，梯度容易下溢为 0，需要 GradScaler；bf16 指数位与 fp32 相同，`enabled=False` 即可。GTX 1650 上 fp16 不一定更快，用不用 AMP 要实测。

In [ ]:
amp_device_10_5 = "cuda" if torch.cuda.is_available() else "cpu"
amp_dtype_10_5 = torch.float16 if amp_device_10_5 == "cuda" else torch.bfloat16
lin_10_5 = nn.Linear(8, 8).to(amp_device_10_5)
x_10_5 = torch.randn(4, 8, device=amp_device_10_5)

with torch.autocast(device_type=amp_device_10_5, dtype=amp_dtype_10_5):
    y_amp_10_5 = lin_10_5(x_10_5)
y_outside_10_5 = lin_10_5(x_10_5)
amp_ok_10_5 = y_amp_10_5.dtype in (torch.float16, torch.bfloat16)
outside_ok_10_5 = y_outside_10_5.dtype == torch.float32

### 练习 10.6　AMP 流程与选择题

正确顺序：`zero_grad → scale(loss).backward() → unscale_ → clip → scaler.step → scaler.update`。`scale` 进、`unscale` 出，`scaler.step` 在检测到 inf/nan 时会跳过参数更新，避免污染权重。

In [ ]:
answer_10_6a = "A"
answer_10_6b = "B"
answer_10_6c = "B"

## 第十一部分　综合实战（手册第 20 章 / 附录 A、B）

### 练习 11.1　训练一个 MLP 学会 XOR

标准四步循环：`zero_grad → 前向+loss → backward → step`。XOR 线性不可分，隐藏层 + 非线性是必要条件；没有 Tanh 的话无论训多久都只能拟合一条直线。loss 不降时先查：忘了 zero_grad、标签 dtype、激活函数、学习率。

In [ ]:
torch.manual_seed(8)
X_11_1 = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_11_1 = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

model_11_1 = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
opt_11_1 = torch.optim.Adam(model_11_1.parameters(), lr=0.05)

losses_11_1 = []
for _ in range(1500):
    opt_11_1.zero_grad()
    loss = nn.BCEWithLogitsLoss()(model_11_1(X_11_1), y_11_1)
    loss.backward()
    opt_11_1.step()
    losses_11_1.append(loss.item())

final_loss_11_1 = losses_11_1[-1]
preds = (torch.sigmoid(model_11_1(X_11_1)) > 0.5).float()
accuracy_11_1 = float((preds == y_11_1).float().mean())

### 练习 11.2　补全因果 Transformer 块

Pre-LN（归一化在子层之前、残差主干干净）是现代大模型标配，训练更稳；后置 LN 需要 warmup。因果 mask 保证第 t 个 token 只看 ≤t。MiniMind 的 TransformerBlock 就是这个骨架：RMSNorm + 多头注意力 + 残差 + SwiGLU FFN + 残差。

In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, d=32, heads=4):
        super().__init__()
        self.norm1 = nn.RMSNorm(d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.norm2 = nn.RMSNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.SiLU(), nn.Linear(4 * d, d))
    def forward(self, x):
        L = x.size(1)
        mask = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), diagonal=1)
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, attn_mask=mask, need_weights=False)
        x = x + a
        return x + self.ffn(self.norm2(x))

block_11_2 = TinyBlock()
block_11_2.eval()
x_11_2 = torch.randn(2, 6, 32)
out_11_2 = block_11_2(x_11_2)

x2_11_2 = x_11_2.clone()
x2_11_2[:, -1] += 5.0
out2_11_2 = block_11_2(x2_11_2)
causal_11_2 = bool(torch.allclose(out_11_2[:, :-1], out2_11_2[:, :-1], atol=1e-6))

### 练习 11.3　报错诊断（选择题）

这五条覆盖了 PyTorch 报错的绝大多数：形状、dtype、计算图生命周期、梯度上下文、设备。遇到报错按「形状 → dtype → 设备 → 计算图」的顺序排查，比盲目搜索快得多。

In [ ]:
answer_11_3a = "B"
answer_11_3b = "B"
answer_11_3c = "A"
answer_11_3d = "B"
answer_11_3e = "A"

### 练习 11.4　断点续训：完整检查点

续训要存「全套」：model、optimizer（动量/二阶矩）、scheduler、step；只存权重会让 Adam 从零状态重新热身，恢复后的前几步明显抖动。生产里再加原子写（先写 `.tmp` 再 `os.replace`），防止写到一半崩溃损坏旧检查点。

In [ ]:
torch.manual_seed(9)
X_11_4 = torch.randn(8, 4)
y_11_4 = torch.randn(8, 1)

def make_model_11_4():
    return nn.Linear(4, 1)

def train_11_4(model, opt, steps):
    for _ in range(steps):
        opt.zero_grad()
        F.mse_loss(model(X_11_4), y_11_4).backward()
        opt.step()

m_cont_11_4 = make_model_11_4()
o_cont_11_4 = torch.optim.AdamW(m_cont_11_4.parameters(), lr=1e-2)
train_11_4(m_cont_11_4, o_cont_11_4, 5)

ckpt_dir_11_4 = tempfile.mkdtemp()
ckpt_path_11_4 = os.path.join(ckpt_dir_11_4, "ckpt.pt")
torch.save({
    "model": m_cont_11_4.state_dict(),
    "optim": o_cont_11_4.state_dict(),
    "step": 5,
}, ckpt_path_11_4)

train_11_4(m_cont_11_4, o_cont_11_4, 5)

m_new_11_4 = make_model_11_4()
o_new_11_4 = torch.optim.AdamW(m_new_11_4.parameters(), lr=1e-2)
ckpt_11_4 = torch.load(ckpt_path_11_4, map_location="cpu", weights_only=True)
m_new_11_4.load_state_dict(ckpt_11_4["model"])
o_new_11_4.load_state_dict(ckpt_11_4["optim"])
train_11_4(m_new_11_4, o_new_11_4, 5)

resume_matches_11_4 = bool(torch.allclose(
    m_new_11_4.weight, m_cont_11_4.weight, atol=1e-6))
optim_state_restored_11_4 = all(
    torch.allclose(o_new_11_4.state[p_new]["exp_avg"],
                   o_cont_11_4.state[p_cont]["exp_avg"])
    for p_new, p_cont in zip(m_new_11_4.parameters(), m_cont_11_4.parameters())
)